# Pipeline Improvement Plan 1 -- Phase 3a: hard mining and multi-peak selection

1. Converts SoccerNet-Tracking `test` (= the 49 GSR `test` clips c5c trained on) to the GSR layout, so
   `pool` is 82 clips as in c5c's training (the Kaggle GSR copy has train/valid only).
2. Caches the main detector on train / pool / select / test.
3. Replays the live system (Phase 1) on every split, keeping c5c's top-5 heatmap peaks per call.
4. Mines the specialist's own failures from train + pool (hard positives B3/B5/B6, false c5c peaks,
   main-detector false balls) into `mined.csv`, for the Phase 3b dataset.
5. Multi-peak selection (highest peak inside the Phase 1 gate radius), chosen on `select`, frozen on `test`.
6. Split integrity with the new clips (same game / overlapping clip / near-duplicate frames).

**Settings:** Accelerator **GPU T4 x2**, Internet **on** (pip installs ultralytics>=8.4).

**Inputs:** soccernetgsr, soccernettracking, phase0-models.

**Disk:** nothing large is written to `/kaggle/working` (clips are symlinked); outputs are a few hundred MB.

**Time:** convert ~2 min, cache ~30-45 min (2 GPUs), replay ~20-30 min (2 GPUs), multi-peak ~15 min,
report + integrity ~10 min. `SMOKE = True` for a ~10-minute dry run (1 sequence per split, 150 frames).

Generated 2026-11-12 15:10 by `scripts/pipeline_improvement_1/phase4/build_phase4_notebook.py` (commit `cc365ad + uncommitted changes`).

| kit file | sha256 |
|---|---|
| `scripts/ball_temporal/gsr_io.py` | `cced1eb2d11c` |
| `scripts/ball_temporal/cache_main_detections.py` | `fb71aabc3e43` |
| `scripts/ball_temporal/heatmap_model.py` | `cadf23b723de` |
| `scripts/ball_temporal/build_ball_temporal_dataset.py` | `fa53456d45cf` |
| `scripts/ball_eval/eval_ball_temporal.py` | `85cdf684c521` |
| `gsr_pipeline/ball_gate.py` | `b1c8aa59fe6f` |
| `scripts/pipeline_improvement_1/phase1/phase1_sweep.py` | `472351f5ebf7` |
| `scripts/pipeline_improvement_1/phase0/phase0_report.py` | `07c7ec58ad4b` |
| `scripts/pipeline_improvement_1/phase3/convert_sntracking.py` | `31cd929ab5bf` |
| `scripts/pipeline_improvement_1/phase3/phase3a_mine.py` | `7fa6e6c7bd22` |

In [ ]:
# Settings. Input paths confirmed by the user.
import os, shutil, subprocess, sys
from pathlib import Path

SMOKE = False

GSR_INPUT   = "/kaggle/input/datasets/sabihalii/soccernetgsr"
TRACK_INPUT = "/kaggle/input/datasets/sabihalii/soccernettracking"
MODELS_DIR  = "/kaggle/input/datasets/sabihalii/phase0-models"

KIT  = "/kaggle/working/kit/cv_service"
WORK = Path("/kaggle/working/" + ("phase3a_smoke" if SMOKE else "phase3a"))
SNT  = Path("/kaggle/working/sntracking_gsr")          # labels + img1 symlinks only
SRC  = f"{GSR_INPUT},{SNT}"
CACHE = WORK / "main_cache"
WORK.mkdir(parents=True, exist_ok=True)


def _find(root, name):
    hits = sorted(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}: check the path")
    return str(hits[0])


MAIN, C5C = _find(MODELS_DIR, "main_best.pt"), _find(MODELS_DIR, "c5c_best.pt")


def STREAM(cmd):
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"stopped (exit {p.returncode}); fix the cause above and re-run this cell")


def PARALLEL(cmds, logs):
    """One process per GPU; prints each log's last line every minute and fails if any process fails."""
    import time
    procs = [subprocess.Popen([str(c) for c in cmd], stdout=open(log, "w"), stderr=subprocess.STDOUT)
             for cmd, log in zip(cmds, logs)]
    while any(p.poll() is None for p in procs):
        time.sleep(60)
        for log in logs:
            lines = Path(log).read_text().splitlines()
            print(Path(log).name, "|", lines[-1] if lines else "(no output yet)", flush=True)
    for log, p in zip(logs, procs):
        if p.returncode:
            print(Path(log).read_text()[-3000:])
    assert not any(p.returncode for p in procs), [p.returncode for p in procs]


def SHOW(path):
    from IPython.display import Markdown, display
    p = Path(path)
    display(Markdown(p.read_text(encoding="utf-8")) if p.exists() else Markdown(f"*{p} not written yet*"))


def DISK():
    u = shutil.disk_usage("/kaggle/working")
    print(f"/kaggle/working: {u.used / 1e9:.1f} GB used, {u.free / 1e9:.1f} GB free")


print("main:", MAIN, "| c5c:", C5C, "| work:", WORK, "| smoke:", SMOKE)
DISK()

## 0. Kit files

These cells write the code to `/kaggle/working/kit/cv_service`. Run them all; don't edit them here (edit the repo and regenerate the notebook).

In [ ]:
import os
for d in ['gsr_pipeline', 'scripts/ball_eval', 'scripts/ball_temporal', 'scripts/pipeline_improvement_1/phase0', 'scripts/pipeline_improvement_1/phase1', 'scripts/pipeline_improvement_1/phase3']:
    os.makedirs(f'/kaggle/working/kit/cv_service/{d}', exist_ok=True)
print('kit folders ready')

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/gsr_io.py
"""
gsr_io.py -- SoccerNet-GSR readers and the sequence splits for the temporal ball specialist
(docs/path_b/plans/BALL_TEMPORAL_PLAN.md §3, §6).

Standalone (numpy, opencv-python only), so the scripts in this folder run on Kaggle as is.

Splits, by sequence (decision A, 2026-10-12):
  train   all GSR `train` sequences. The main detector was trained on them, so their frames give
          easy positives and negatives only, never hard examples.
  pool    `valid` sequences outside test and select, plus every GSR `test` sequence (added
          2026-10-15). The main detector never trained on either: the source of hard examples.
  select  10 `valid` sequences, evenly spaced through the non-test ones: model selection and
          early stopping.
  test    the 15 held-out `valid` sequences: final gates only.

Sources: a directory searched recursively for sequence folders (Labels-GameState.json + img1/),
or failing that the split zips (train.zip / valid.zip), read without extracting.
"""

from __future__ import annotations

import json
import zipfile
from dataclasses import dataclass
from pathlib import Path, PurePosixPath

import numpy as np

TEST_SEQS = frozenset({21, 22, 23, 26, 30, 33, 36, 40, 44, 47, 50, 54, 80, 86, 90})
N_SELECT = 10
BALL_CAT = 4
PERSON_CATS = frozenset({1, 2, 3, 7})


@dataclass(frozen=True)
class SeqRef:
    name: str          # SNGS-060
    src_split: str     # train | valid | gsrtest (the GSR folder it came from)
    kind: str          # dir | zip
    container: str     # sequence dir, or zip path
    prefix: str        # inside-zip prefix of the sequence folder ("" for dir)
    split: str = ""    # train | pool | select | test, filled in by discover()

    @property
    def number(self) -> int:
        return int(self.name.split("-")[-1])


def _gsr_split_of(parts) -> str | None:
    low = [p.lower() for p in parts]
    if any("train" in p for p in low):
        return "train"
    if any("valid" in p for p in low):
        return "valid"
    if any(p == "test" or p.startswith("test.") for p in low):
        return "gsrtest"            # the GSR test split; not our held-out `test` (valid sequences)
    return None


def select_seqs(valid_numbers) -> set[int]:
    """N_SELECT sequences evenly spaced through the sorted non-test valid sequences."""
    rest = sorted(n for n in valid_numbers if n not in TEST_SEQS)
    if len(rest) <= N_SELECT:
        return set(rest)
    return {rest[int((i + 0.5) * len(rest) / N_SELECT)] for i in range(N_SELECT)}


def discover(src: Path) -> list[SeqRef]:
    """Sequence folders under src, else sequences inside zips under src. One entry per name.
    src may list several roots separated by commas (e.g. GSR + converted SoccerNet-Tracking clips)."""
    found: dict[str, SeqRef] = {}
    for root in str(src).split(","):
        _discover_root(Path(root), found)
    return _assign_splits(found)


def _discover_root(src: Path, found: dict) -> None:
    n0 = len(found)
    for lab in sorted(Path(src).rglob("Labels-GameState.json")):
        seq = lab.parent
        gsr = _gsr_split_of(seq.relative_to(src).parts)
        if gsr and seq.name not in found and (seq / "img1").is_dir():
            found[seq.name] = SeqRef(seq.name, gsr, "dir", str(seq), "")
    if len(found) == n0:
        for zp in sorted(Path(src).rglob("*.zip")):
            with zipfile.ZipFile(zp) as z:
                for n in z.namelist():
                    if not n.endswith("Labels-GameState.json"):
                        continue
                    pre = str(PurePosixPath(n).parent)
                    name = PurePosixPath(pre).name
                    gsr = _gsr_split_of(PurePosixPath(pre).parts) or _gsr_split_of([zp.name])
                    if gsr and name.startswith("SNGS") and name not in found:
                        found[name] = SeqRef(name, gsr, "zip", str(zp), pre)


def _assign_splits(found: dict) -> list[SeqRef]:
    sel = select_seqs(r.number for r in found.values() if r.src_split == "valid")

    def split_of(r: SeqRef) -> str:
        if r.src_split == "train":
            return "train"
        if r.src_split == "gsrtest":
            return "pool"
        return "test" if r.number in TEST_SEQS else "select" if r.number in sel else "pool"

    refs = [SeqRef(r.name, r.src_split, r.kind, r.container, r.prefix, split_of(r)) for r in found.values()]
    return sorted(refs, key=lambda s: s.number)


class SeqReader:
    """Reads one sequence's labels and frames from a folder or a zip (one per process)."""

    def __init__(self, ref: SeqRef):
        self.ref = ref
        self._zip = zipfile.ZipFile(ref.container) if ref.kind == "zip" else None

    def read(self, rel: str) -> bytes:
        if self._zip is not None:
            return self._zip.read(f"{self.ref.prefix}/{rel}")
        return (Path(self.ref.container) / rel).read_bytes()

    def labels(self) -> dict:
        return json.loads(self.read("Labels-GameState.json"))

    def close(self):
        if self._zip is not None:
            self._zip.close()


def decode(buf: bytes):
    """JPEG bytes -> BGR image. cv2.imdecode releases the GIL, so threads can share it."""
    import cv2
    img = cv2.imdecode(np.frombuffer(buf, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("cannot decode frame")
    return img


def parse_labels(d: dict):
    """-> im_dir, {frame: file_name}, {frame: [(cx,cy,w,h,track_id)]}, persons, line points, (W,H)."""
    info = d.get("info", {})
    im_dir = info.get("im_dir", "img1")
    W, H = 1920, 1080
    files, id2f = {}, {}
    for im in d["images"]:
        f = int(Path(im["file_name"]).stem)
        files[f] = im["file_name"]
        id2f[im["image_id"]] = f
        W, H = im.get("width", W), im.get("height", H)
    balls, persons, lines = {}, {}, {}
    for a in d["annotations"]:
        f = id2f.get(a["image_id"])
        if f is None:
            continue
        cat = a.get("category_id")
        if cat == BALL_CAT or cat in PERSON_CATS:
            b = a["bbox_image"]
            row = (float(b["x_center"]), float(b["y_center"]), float(b["w"]), float(b["h"]), a.get("track_id"))
            (balls if cat == BALL_CAT else persons).setdefault(f, []).append(row)
        elif a.get("supercategory") == "pitch":
            pts = [(p["x"] * W, p["y"] * H) for seg in a.get("lines", {}).values() for p in seg]
            lines[f] = [(x, y) for x, y in pts if 0 <= x < W and 0 <= y < H]
    return im_dir, files, balls, persons, lines, (W, H)

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/cache_main_detections.py
"""
cache_main_detections.py -- step 1 of docs/path_b/plans/BALL_TEMPORAL_PLAN.md §9: run the main
athlete/ball detector over every frame and cache its ball detections per sequence.

The cache feeds step 2 (the temporal dataset builder): it marks hard frames (GSR has a ball,
the main detector missed it) and centres crops on the last prediction, as the runtime does.

Standalone apart from gsr_io.py (same folder). Needs ultralytics>=8.4.0 (YOLO26).

  list     print the sequences per split (train / pool / select / test); no GPU
  run      detect and cache; --shard i --nshards n splits the sequences across GPUs
  summary  main-detector recall per split and hard-frame counts, from the cache; no GPU

Cache: <out>/<split>/<SNGS-xxx>.json =
  {"meta": {weights, imgsz, conf, top}, "frames": {"<frame>": {"balls": [[cx, cy, w, h, conf], ...],
                                                               "gt": [[cx, cy, size], ...],
                                                               "athletes": [[cx, cy, w, h, conf], ...]}}}
  (athletes only with --athletes)
balls: every ball detection with conf >= --conf, highest first, at most --top, in native pixels.
Finished sequences are skipped, and each file is written atomically, so a stopped run resumes.

    python cache_main_detections.py run --src /kaggle/input/datasets/sabihalii/soccernetgsr \
        --out /kaggle/working/main_cache --weights /kaggle/working/kit/best.pt --device 0 --half
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from gsr_io import SeqReader, decode, discover, parse_labels  # noqa: E402

SPLITS = ("train", "pool", "select", "test")
MATCH_PX = 15.0      # ball match radius, as the gates (plan §8)
HARD_CONF = 0.15     # a frame is hard when no ball at this confidence is within MATCH_PX of GT


def _balls(result, cls: int, top: int) -> list[list[float]]:
    b = result.boxes
    if b is None or len(b) == 0:
        return []
    keep = b.cls.cpu().numpy().astype(int) == cls
    xywh, conf = b.xywh.cpu().numpy()[keep], b.conf.cpu().numpy()[keep]
    order = np.argsort(-conf)[:top]
    return [[round(float(v), 2) for v in (*xywh[i], conf[i])] for i in order]


def _athletes(result, ball_cls: int, top: int, min_conf: float) -> list[list[float]]:
    """Every non-ball box at conf >= min_conf, highest first, at most top: [cx, cy, w, h, conf]."""
    b = result.boxes
    if b is None or len(b) == 0:
        return []
    conf = b.conf.cpu().numpy()
    keep = (b.cls.cpu().numpy().astype(int) != ball_cls) & (conf >= min_conf)
    xywh, conf = b.xywh.cpu().numpy()[keep], conf[keep]
    order = np.argsort(-conf)[:top]
    return [[round(float(v), 1) for v in (*xywh[i], conf[i])] for i in order]


def _refs(a):
    refs = [r for r in discover(a.src) if r.split in a.splits]
    if a.seqs:
        refs = [r for r in refs if r.name in a.seqs]
    return refs


def stage_list(a):
    refs = discover(a.src)
    for s in SPLITS:
        names = [r.name for r in refs if r.split == s]
        print(f"{s:6s} {len(names):3d}  {' '.join(n.replace('SNGS-', '') for n in names)}")


def stage_run(a):
    from ultralytics import YOLO
    model = YOLO(a.weights)
    names = {v: k for k, v in model.names.items()}
    if "ball" not in names:
        raise SystemExit(f"model has no 'ball' class: {model.names}")
    cls = names["ball"]
    kw = dict(imgsz=a.imgsz, conf=a.conf, iou=0.7, max_det=300, device=a.device, verbose=False)
    if a.half:
        kw["half"] = True                         # only when asked: Ultralytics 8.4 warns on any `half`
    meta = {"weights": str(a.weights), "imgsz": a.imgsz, "conf": a.conf, "top": a.top,
            "athletes": a.athletes, "athlete_conf": a.athlete_conf}

    refs = [r for i, r in enumerate(_refs(a)) if i % a.nshards == a.shard]
    tag = f"[gpu{a.device}]"
    print(f"{tag} {len(refs)} sequences", flush=True)
    pool = ThreadPoolExecutor(a.decode_threads)
    t_all = time.time()
    for k, ref in enumerate(refs, 1):
        out = a.out / ref.split / f"{ref.name}.json"
        if out.exists():
            print(f"{tag} {k}/{len(refs)} {ref.name}: cached", flush=True)
            continue
        out.parent.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        rd = SeqReader(ref)
        im_dir, files, gt, *_ = parse_labels(rd.labels())
        frames = sorted(files)[:a.max_frames] if a.max_frames else sorted(files)
        rec = {}
        for i in range(0, len(frames), a.batch):
            chunk = frames[i:i + a.batch]
            bufs = [rd.read(f"{im_dir}/{files[f]}") for f in chunk]   # zip reads stay on one thread
            imgs = list(pool.map(decode, bufs))
            for f, r in zip(chunk, model.predict(imgs, **kw)):
                rec[str(f)] = {"balls": _balls(r, cls, a.top),
                               **({"athletes": _athletes(r, cls, 40, a.athlete_conf)} if a.athletes else {}),
                               "gt": [[round(b[0], 2), round(b[1], 2), round(max(b[2], b[3]), 2)]
                                      for b in gt.get(f, [])]}
        rd.close()
        tmp = out.with_suffix(".tmp")
        tmp.write_text(json.dumps({"meta": meta, "frames": rec}))
        tmp.replace(out)
        dt = time.time() - t0
        print(f"{tag} {k}/{len(refs)} {ref.split}/{ref.name}: {len(rec)} frames in {dt:.0f}s "
              f"({1000 * dt / max(len(rec), 1):.0f} ms/frame), elapsed {(time.time() - t_all) / 60:.1f} min",
              flush=True)
    print(f"{tag} done in {(time.time() - t_all) / 60:.1f} min", flush=True)


def _hit(balls, gts, conf) -> bool:
    return any(b[4] >= conf and np.hypot(b[0] - g[0], b[1] - g[1]) <= MATCH_PX for b in balls for g in gts)


def stage_summary(a):
    lines = ["# Main-detector cache summary", "",
             f"Hard frame: GSR has a ball and no main-detector ball at conf >= {HARD_CONF} is within "
             f"{MATCH_PX:.0f} px of it.", "",
             "| split | sequences | frames | frames with GT ball | main recall (top ball) | hard frames | hard share |",
             "|---|---|---|---|---|---|---|"]
    for s in SPLITS:
        files = sorted((a.out / s).glob("SNGS-*.json"))
        if not files:
            continue
        n_fr = n_gt = n_top = n_hard = 0
        for fp in files:
            for v in json.loads(fp.read_text())["frames"].values():
                n_fr += 1
                if not v["gt"]:
                    continue
                n_gt += 1
                n_top += _hit(v["balls"][:1], v["gt"], HARD_CONF)
                n_hard += not _hit(v["balls"], v["gt"], HARD_CONF)
        lines.append(f"| {s} | {len(files)} | {n_fr} | {n_gt} | {n_top / max(n_gt, 1):.2%} | "
                     f"{n_hard} | {n_hard / max(n_gt, 1):.2%} |")
    lines += ["", "If train recall is clearly above pool/select/test, the detector memorised train, "
                  "and hard examples must come from pool only (plan §6, decision A)."]
    text = "\n".join(lines)
    print(text)
    (a.out / "summary.md").write_text(text)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["list", "run", "summary"])
    ap.add_argument("--src", type=Path, help="GSR source: sequence dirs or the split zips")
    ap.add_argument("--out", type=Path, default=Path("/kaggle/working/main_cache"))
    ap.add_argument("--weights", default="/kaggle/working/kit/best.pt", help="main athlete/ball detector")
    ap.add_argument("--splits", nargs="+", default=list(SPLITS), choices=SPLITS)
    ap.add_argument("--seqs", nargs="*", default=[], help="only these sequences, e.g. SNGS-060 (smoke tests)")
    ap.add_argument("--imgsz", type=int, default=1280, help="1280 = the 736x1280 runtime input after letterbox")
    ap.add_argument("--conf", type=float, default=0.05, help="lowest ball confidence kept in the cache")
    ap.add_argument("--top", type=int, default=5, help="most ball detections kept per frame")
    ap.add_argument("--athletes", action="store_true",
                    help="also cache athlete boxes (\"athletes\": [[cx, cy, w, h, conf], ...], up to 40) for the ball tracker")
    ap.add_argument("--athlete_conf", type=float, default=0.3, help="lowest athlete confidence kept with --athletes")
    ap.add_argument("--device", default="0")
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--half", action="store_true", help="FP16 inference (GPU only)")
    ap.add_argument("--decode_threads", type=int, default=2, help="JPEG decode threads per process")
    ap.add_argument("--max_frames", type=int, default=0, help="first N frames per sequence (smoke tests)")
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--nshards", type=int, default=1)
    a = ap.parse_args()
    if a.stage in ("list", "run") and not a.src:
        ap.error("--src is required")
    {"list": stage_list, "run": stage_run, "summary": stage_summary}[a.stage](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/heatmap_model.py
"""
heatmap_model.py -- the temporal ball specialist (docs/path_b/plans/BALL_TEMPORAL_PLAN.md §4-§5):
model, input construction, targets, loss and decode. Shared by training, evaluation and export.

Input: 3 frames t-2, t-1, t of one 640x640 window, as uint8 RGB (B, 9, 640, 640) in that order.
  9 channels: the three RGB crops, ImageNet-normalised.
  5 channels: RGB at t (normalised) + grey(t) - grey(t-1) and grey(t-1) - grey(t-2), scaled by 1/0.226.
Output: heatmap logits (B, 1, 320, 320) and sub-pixel offset (B, 2, 320, 320), stride 2.
A ball at crop pixel (x, y) peaks at heatmap cell (floor(x/2), floor(y/2)) with offset (x/2, y/2) - cell.

Needs torch and torchvision.
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

STRIDE = 2
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
DIFF_SCALE = 1 / 0.226
PRIOR = 0.1                       # initial heatmap sigmoid output, CenterNet-style


def build_input(frames: torch.Tensor, channels: int) -> torch.Tensor:
    """uint8 (B, 9, H, W) frames t-2|t-1|t RGB -> float model input with `channels` (5 or 9) channels."""
    x = frames.float() / 255
    mean = x.new_tensor(MEAN).view(1, 3, 1, 1)
    std = x.new_tensor(STD).view(1, 3, 1, 1)
    f2, f1, f0 = x[:, 0:3], x[:, 3:6], x[:, 6:9]
    if channels == 9:
        return torch.cat([(f - mean) / std for f in (f2, f1, f0)], 1)
    if channels == 5:
        g = [f.mean(1, keepdim=True) for f in (f2, f1, f0)]
        return torch.cat([(f0 - mean) / std, (g[2] - g[1]) * DIFF_SCALE, (g[1] - g[0]) * DIFF_SCALE], 1)
    raise ValueError(f"channels must be 5 or 9, got {channels}")


class _Up(nn.Module):
    def __init__(self, c_in, c_skip, c_out):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(c_in + c_skip, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], 1))


class BallHeatmapNet(nn.Module):
    """ResNet-18 encoder (first conv widened to `channels`), U-Net decoder to a stride-2 heatmap + offset."""

    def __init__(self, channels: int = 9, pretrained: bool = True):
        super().__init__()
        from torchvision.models import ResNet18_Weights, resnet18
        r = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        w = r.conv1.weight.data                                   # (64, 3, 7, 7)
        conv1 = nn.Conv2d(channels, 64, 7, 2, 3, bias=False)
        with torch.no_grad():
            if channels == 9:      # each frame gets the RGB weights, /3 so activations keep their size
                conv1.weight.copy_(torch.cat([w, w, w], 1) / 3)
            else:                  # RGB at t as pretrained; diff channels see a grey image -> summed weights
                g = w.sum(1, keepdim=True)
                conv1.weight.copy_(torch.cat([w, g * 0.5, g * 0.5], 1))
        self.channels = channels
        self.stem = nn.Sequential(conv1, r.bn1, r.relu)            # stride 2, 64
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4   # 4, 8, 16, 32
        self.up4 = _Up(512, 256, 256)
        self.up3 = _Up(256, 128, 128)
        self.up2 = _Up(128, 64, 64)
        self.up1 = _Up(64, 64, 64)                                 # -> stride 2
        self.hm = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(64, 1, 1))
        self.off = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(64, 2, 1))
        nn.init.constant_(self.hm[-1].bias, -2.19)                 # sigmoid ~ PRIOR

    def forward(self, x):
        s2 = self.stem(x)
        s4 = self.l1(self.pool(s2))
        s8 = self.l2(s4)
        s16 = self.l3(s8)
        s32 = self.l4(s16)
        d = self.up1(self.up2(self.up3(self.up4(s32, s16), s8), s4), s2)
        return self.hm(d), self.off(d)


def make_targets(points: torch.Tensor, hm_size: int):
    """points (B, K, 3) crop px (x, y, size), rows with size <= 0 unused ->
    heatmap (B, 1, S, S), peak cell index (B, K), offset (B, K, 2), mask (B, K)."""
    B, K, _ = points.shape
    dev = points.device
    valid = points[..., 2] > 0
    c = points[..., :2] / STRIDE
    cell = c.floor().clamp(0, hm_size - 1)
    off = c - cell
    sigma = (points[..., 2] / STRIDE / 4).clamp(min=1.5)
    ys = torch.arange(hm_size, device=dev, dtype=torch.float32).view(1, 1, hm_size, 1)
    xs = torch.arange(hm_size, device=dev, dtype=torch.float32).view(1, 1, 1, hm_size)
    d2 = (xs - cell[..., 0, None, None]) ** 2 + (ys - cell[..., 1, None, None]) ** 2
    g = torch.exp(-d2 / (2 * sigma[..., None, None] ** 2)) * valid[..., None, None]
    hm = g.amax(1, keepdim=True) if K else torch.zeros(B, 1, hm_size, hm_size, device=dev)
    ind = (cell[..., 1] * hm_size + cell[..., 0]).long()
    return hm, ind, off, valid


def focal_loss(logits: torch.Tensor, gt: torch.Tensor) -> torch.Tensor:
    """CenterNet penalty-reduced focal loss, normalised by the number of peaks (min 1)."""
    p = logits.float().sigmoid().clamp(1e-4, 1 - 1e-4)
    pos = gt.eq(1).float()
    neg = 1 - pos
    pos_loss = torch.log(p) * (1 - p) ** 2 * pos
    neg_loss = torch.log(1 - p) * p ** 2 * (1 - gt) ** 4 * neg
    return -(pos_loss.sum() + neg_loss.sum()) / pos.sum().clamp(min=1)


def offset_loss(off: torch.Tensor, ind: torch.Tensor, off_gt: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    B, _, S, _ = off.shape
    pred = off.float().view(B, 2, S * S).gather(2, ind.clamp(0, S * S - 1)[:, None].expand(-1, 2, -1))
    pred = pred.permute(0, 2, 1)                                   # (B, K, 2)
    m = mask[..., None].float()
    return (F.l1_loss(pred, off_gt, reduction="none") * m).sum() / m.sum().clamp(min=1)


@torch.no_grad()
def decode(logits: torch.Tensor, off: torch.Tensor):
    """-> (B, 3): x, y in crop pixels and confidence of the single strongest peak."""
    B, _, S, _ = logits.shape
    p = logits.float().sigmoid().view(B, -1)
    conf, i = p.max(1)
    iy, ix = (i // S).float(), (i % S).float()
    o = off.float().view(B, 2, -1).gather(2, i[:, None, None].expand(-1, 2, 1))[..., 0]
    return torch.stack([(ix + o[:, 0]) * STRIDE, (iy + o[:, 1]) * STRIDE, conf], 1)

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_temporal/build_ball_temporal_dataset.py
"""
build_ball_temporal_dataset.py -- step 2 of docs/path_b/plans/BALL_TEMPORAL_PLAN.md §9: the
3-frame, point-labelled crop dataset for the temporal ball specialist (plan §4-§6).

Standalone apart from gsr_io.py (same folder). Needs numpy, opencv-python. Reads the main-detector
cache from step 1 (cache_main_detections.py).

  build    write the samples and manifest.csv; resumes per sequence
  stats    counts per split and kind, from manifest.csv
  preview  contact sheets of random samples with their labels drawn, for the spot-check by eye

Per ball-labelled frame t (frames without a ball label give no crops, plan §3):
  * Window: 640x640 native pixels, shift-clamped into the frame (never padded), centred like the
    runtime (plan §6.3): on the main detector's top ball (conf >= --pred_conf) in the most recent
    of frames t-1 .. t-15; failing that, on the GT ball in the most recent labelled frame of that
    range, +-48 px jitter; failing both, no positive crop.
  * Hard frame: GT ball, and no main ball at conf >= --pred_conf within 15 px of it. Marked in
    every split; hard *positives* are only sampled as such outside `train` (decision A).
  * Positives: every hard frame (repeat 3 in pool, 1 elsewhere); other frames every 3rd (t % 3 == 0).
    A ball cut by the window is labelled if its centre and >= 50% of its box are inside, otherwise
    the crop is dropped. If the window misses the ball entirely (centred on a false positive, or
    a long gap), the crop is kept as kind `outside`, with an empty label: the runtime sees it too.
  * Negatives, from positive-sampled frames only: hard (person or pitch-line point, +-64 px) for
    15% of crops and random for 5%; no ball in the window at t, t-1 or t-2.
  * --centre predicted (docs/plans/BALL_IMPROVEMENT_PLAN.md §4.3): the window is centred on the
    ball tracker's prediction instead (gsr_pipeline/ball_tracker.py replayed over main sightings,
    with camera motion from camera_motion_cache.py), falling back to the rule above when there is
    no live track; centre_source "kf". --gap2_repeat k multiplies the repeat of pool hard
    positives whose centre is >= 2 frames old. Defaults reproduce the c5c dataset.
  * Frames t-1, t-2 are cut with the window chosen at t. Before the sequence start they repeat
    frame t (n_prev < 2). The 5% "repeat t" training case is an augmentation in step 3, not here.

Output (<out>/):
  <split>/<SNGS-xxx>/<frame>_<kind>.jpg   one sample: crops t-2 | t-1 | t side by side, 640x1920
  manifest.csv                            one row per sample (columns: MANIFEST_FIELDS)
  parts/<SNGS-xxx>.csv                    per-sequence manifest, written last (= sequence done)
balls: "cx cy size;..." in crop pixels of frame t (size = max(w, h) of the GSR box), empty if none.
main_conf: positives, the highest main-detector confidence within 15 px of the GT ball (0 if none);
negatives and `outside`, the highest main-detector confidence inside the window (0 if none).

    python scripts/ball_temporal/build_ball_temporal_dataset.py build --src data/datasets/SoccerNetGS \
        --cache data/outputs/ball_temporal/main_cache --out data/outputs/ball_temporal/dataset --workers 8
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import sys
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "gsr_pipeline"))     # ball_tracker.py, plain import
from gsr_io import SeqReader, decode, discover, parse_labels  # noqa: E402

SPLITS = ("train", "pool", "select", "test")
CROP = 640
LOOKBACK = 15            # frames searched back for the window centre (= max_miss_frames)
JITTER = 48              # on GT-centred windows only
EASY_STEP = 3            # easy positives: every 3rd frame
HARD_REPEAT = 3          # hard positives in pool, via the manifest's repeat column
NEG_HARD_SHARE = 0.15    # of all crops
NEG_RANDOM_SHARE = 0.05
MIN_VISIBLE = 0.5
MATCH_PX = 15.0
MANIFEST_FIELDS = ["split", "seq", "frame", "kind", "hard", "repeat", "centre_source", "centre_gap",
                   "main_conf", "x0", "y0", "clamped", "n_prev", "n_balls", "ball_size", "ball_dx",
                   "ball_dy", "balls", "file"]


# --------------------------------------------------------------------------- geometry

def clamp_window(cx: float, cy: float, W: int, H: int) -> tuple[int, int, bool]:
    """Top-left of a CROP x CROP window centred at (cx, cy), shifted (never padded) into the frame."""
    x0, y0 = int(round(cx - CROP / 2)), int(round(cy - CROP / 2))
    cx0 = min(max(x0, 0), max(W - CROP, 0))
    cy0 = min(max(y0, 0), max(H - CROP, 0))
    return cx0, cy0, (cx0, cy0) != (x0, y0)


def balls_in_window(balls, x0, y0):
    """-> (points, dropped). points: [(cx, cy, size)] in crop px; dropped if a ball is cut ambiguously."""
    pts = []
    for cx, cy, w, h, _ in balls:
        l, t, r, b = cx - w / 2 - x0, cy - h / 2 - y0, cx + w / 2 - x0, cy + h / 2 - y0
        inter = max(min(r, CROP) - max(l, 0), 0) * max(min(b, CROP) - max(t, 0), 0)
        if inter <= 0:
            continue
        centre_in = 0 <= cx - x0 < CROP and 0 <= cy - y0 < CROP
        if inter / max(w * h, 1e-6) < MIN_VISIBLE or not centre_in:
            return [], True
        pts.append((cx - x0, cy - y0, max(w, h)))
    return pts, False


def overlaps_any_ball(balls, x0, y0, margin=8):
    return any(cx + w / 2 + margin > x0 and cx - w / 2 - margin < x0 + CROP and
               cy + h / 2 + margin > y0 and cy - h / 2 - margin < y0 + CROP
               for cx, cy, w, h, _ in balls)


def _cut(img, x0, y0):
    c = img[y0:y0 + CROP, x0:x0 + CROP]
    if c.shape[:2] != (CROP, CROP):
        raise ValueError(f"frame {img.shape[:2]} smaller than {CROP}px")
    return c


# --------------------------------------------------------------------------- per sequence

def _matched_conf(dets, gts) -> float:
    return max((d[4] for d in dets for g in gts if np.hypot(d[0] - g[0], d[1] - g[1]) <= MATCH_PX),
               default=0.0)


def _conf_in_window(dets, x0, y0) -> float:
    return max((d[4] for d in dets if x0 <= d[0] < x0 + CROP and y0 <= d[1] < y0 + CROP), default=0.0)


def _centre(t, dets, balls, pred_conf, rng):
    """Runtime-like window centre for frame t -> (cx, cy, source, gap) or None."""
    for g in range(1, LOOKBACK + 1):
        d = [b for b in dets.get(t - g, []) if b[4] >= pred_conf]
        if d:
            return d[0][0], d[0][1], "pred", g
    for g in range(1, LOOKBACK + 1):
        if balls.get(t - g):
            b = balls[t - g][0]
            jx, jy = rng.uniform(-JITTER, JITTER, 2)
            return b[0] + jx, b[1] + jy, "gt", g
    return None


def _tracker_centres(files, dets, motion, pred_conf, tracker_cfg: dict | None) -> dict:
    """--centre predicted: {t: (cx, cy, gap)} = the ball tracker's prediction at t (before it sees
    frame t), replayed over main-detector sightings only (conf >= pred_conf), for frames 1..LOOKBACK
    after the last accepted sighting. Same tracker as the runtime (gsr_pipeline/ball_tracker.py)."""
    import ball_tracker as bt
    cfg = bt.TrackerConfig(**{**(tracker_cfg or {}), "min_conf_main": pred_conf, "init_conf": pred_conf})
    tr = bt.BallTracker(cfg)
    out = {}
    for t in sorted(files):
        pred = tr.predict(t, motion.get(t))
        gap = tr.gap if tr.last_meas is not None else None
        if pred is not None and gap is not None and 1 <= gap <= LOOKBACK:
            out[t] = (pred[0], pred[1], gap)
        tr.update([bt.BallCandidate(d[0], d[1], d[4]) for d in dets.get(t, []) if d[4] >= pred_conf])
    return out


def process_sequence(job):
    import cv2
    ref, cache_dir, out, seed, pred_conf, quality, centre_mode, motion_dir, gap2_repeat, tracker_cfg = job
    part = Path(out) / "parts" / f"{ref.name}.csv"
    if part.exists():
        with open(part, newline="") as f:
            return ref.name, list(csv.DictReader(f)), True
    cache = json.loads((Path(cache_dir) / ref.split / f"{ref.name}.json").read_text())["frames"]
    dets = {int(k): v["balls"] for k, v in cache.items()}       # highest confidence first
    reader = SeqReader(ref)
    try:
        im_dir, files, balls, persons, lines, (W, H) = parse_labels(reader.labels())
        kf = {}
        if centre_mode == "predicted":
            import camera_motion_cache as cmc
            motion = cmc.load(Path(motion_dir), ref) if motion_dir else {}
            kf = _tracker_centres(files, dets, motion, pred_conf, tracker_cfg)
        seq_dir = Path(out) / ref.split / ref.name
        seq_dir.mkdir(parents=True, exist_ok=True)
        frames_cache: dict[int, np.ndarray] = {}

        def frame(f):
            if f not in frames_cache:
                frames_cache[f] = decode(reader.read(f"{im_dir}/{files[f]}"))
            return frames_cache[f]

        rows = []
        for t in sorted(files):
            bt = balls.get(t)
            if not bt:
                continue
            gts = [(b[0], b[1]) for b in bt]
            hard = _matched_conf([d for d in dets.get(t, []) if d[4] >= pred_conf], gts) == 0.0
            sample_hard = hard and ref.split != "train"
            if not sample_hard and t % EASY_STEP:
                continue
            rng = np.random.default_rng([seed, ref.number, t])
            jobs = []        # (kind, x0, y0, clamped, points, meta)

            c = None
            if centre_mode == "predicted" and t in kf:
                c = (kf[t][0], kf[t][1], "kf", kf[t][2])
            if c is None:
                c = _centre(t, dets, balls, pred_conf, rng)
            if c is not None:
                x0, y0, clamped = clamp_window(c[0], c[1], W, H)
                pts, dropped = balls_in_window(bt, x0, y0)
                if not dropped:
                    kind = ("hard" if sample_hard else "easy") if pts else "outside"
                    conf = _matched_conf(dets.get(t, []), gts) if pts else _conf_in_window(dets.get(t, []), x0, y0)
                    rep = HARD_REPEAT if kind == "hard" and ref.split == "pool" else 1
                    if kind == "hard" and ref.split == "pool" and int(c[3]) >= 2:
                        rep *= gap2_repeat                # plan §4.3: more balls far from the last sighting
                    jobs.append((kind, x0, y0, clamped, pts,
                                 dict(centre_source=c[2], centre_gap=c[3], main_conf=conf, repeat=rep)))

            r = rng.random()
            neg_p = 1 / (1 - NEG_HARD_SHARE - NEG_RANDOM_SHARE) - 1     # negatives per positive
            if r < neg_p:
                cands = [(p[0], p[1]) for p in persons.get(t, [])] + lines.get(t, [])
                is_hard = rng.random() < NEG_HARD_SHARE / (NEG_HARD_SHARE + NEG_RANDOM_SHARE) and bool(cands)
                near = bt + balls.get(t - 1, []) + balls.get(t - 2, [])
                for _ in range(20):
                    if is_hard:
                        cx, cy = cands[rng.integers(len(cands))]
                        cx, cy = cx + rng.uniform(-64, 64), cy + rng.uniform(-64, 64)
                    else:
                        cx, cy = rng.uniform(0, W), rng.uniform(0, H)
                    x0, y0, clamped = clamp_window(cx, cy, W, H)
                    if not overlaps_any_ball(near, x0, y0):
                        jobs.append(("neg_hard" if is_hard else "neg_random", x0, y0, clamped, [],
                                     dict(centre_source="", centre_gap="", repeat=1,
                                          main_conf=_conf_in_window(dets.get(t, []), x0, y0))))
                        break

            if not jobs:
                continue
            prev = [f for f in (t - 2, t - 1) if f in files]
            idx = [t - 2 if t - 2 in files else t, t - 1 if t - 1 in files else t, t]
            imgs = [frame(f) for f in idx]
            for f in [k for k in frames_cache if k < t - 2]:
                del frames_cache[f]
            for kind, x0, y0, clamped, pts, meta in jobs:
                name = f"{t:06d}_{kind}.jpg"
                tile = np.concatenate([_cut(im, x0, y0) for im in imgs], axis=1)
                ok, enc = cv2.imencode(".jpg", tile, [cv2.IMWRITE_JPEG_QUALITY, quality])
                if not ok:
                    raise RuntimeError(f"jpeg encode failed: {ref.name} {name}")
                (seq_dir / name).write_bytes(enc.tobytes())
                p0 = pts[0] if pts else None
                rows.append(dict(split=ref.split, seq=ref.name, frame=t, kind=kind, hard=int(hard),
                                 x0=x0, y0=y0, clamped=int(clamped), n_prev=len(prev), n_balls=len(pts),
                                 ball_size=round(p0[2], 1) if p0 else "",
                                 ball_dx=round(p0[0] - CROP / 2, 1) if p0 else "",
                                 ball_dy=round(p0[1] - CROP / 2, 1) if p0 else "",
                                 balls=";".join(f"{x:.1f} {y:.1f} {s:.1f}" for x, y, s in pts),
                                 file=f"{ref.split}/{ref.name}/{name}",
                                 **{**meta, "main_conf": round(meta["main_conf"], 3)}))
        part.parent.mkdir(parents=True, exist_ok=True)
        tmp = part.with_suffix(".tmp")
        with open(tmp, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)
            w.writeheader()
            w.writerows(rows)
        tmp.replace(part)
        return ref.name, rows, False
    finally:
        reader.close()


# --------------------------------------------------------------------------- stages

def stage_build(a):
    refs = [r for r in discover(a.src) if r.split in a.splits]
    if a.seqs:
        refs = [r for r in refs if r.name in a.seqs]
    missing = [r.name for r in refs if not (a.cache / r.split / f"{r.name}.json").exists()]
    if missing:
        raise SystemExit(f"no main-detector cache for {len(missing)} sequences, e.g. {missing[:3]}; run step 1")
    print(f"{len(refs)} sequences, {a.workers} workers", flush=True)
    a.out.mkdir(parents=True, exist_ok=True)
    tracker_cfg = None
    if a.tracker_config:
        tracker_cfg = json.loads(a.tracker_config.read_text()).get("tracker")
    if a.centre == "predicted":
        print(f"window centre: ball tracker prediction (motion {a.motion}, tracker config {a.tracker_config})", flush=True)
    jobs = [(r, str(a.cache), str(a.out), a.seed, a.pred_conf, a.jpeg_quality, a.centre,
             str(a.motion) if a.motion else None, a.gap2_repeat, tracker_cfg) for r in refs]
    t0, rows = time.time(), []
    with Pool(a.workers) as pool:
        for i, (name, r, cached) in enumerate(pool.imap_unordered(process_sequence, jobs), 1):
            rows.extend(r)
            print(f"[{i}/{len(refs)}] {name}: {len(r)} samples{' (cached)' if cached else ''}  "
                  f"({time.time() - t0:.0f}s)", flush=True)
    order = {s: i for i, s in enumerate(SPLITS)}
    rows.sort(key=lambda r: (order[r["split"]], r["seq"], int(r["frame"]), r["kind"]))
    with open(a.out / "manifest.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)
        w.writeheader()
        w.writerows(rows)
    print(f"wrote {a.out / 'manifest.csv'} ({len(rows)} samples) in {(time.time() - t0) / 60:.1f} min\n")
    stage_stats(a)


def stage_stats(a):
    with open(a.out / "manifest.csv", newline="") as f:
        rows = list(csv.DictReader(f))
    kinds = ("hard", "easy", "outside", "neg_hard", "neg_random")
    lines = ["| split | samples | " + " | ".join(kinds) + " | hard frames sampled (any kind) | "
             "with repeat | GT-centred |", "|---" * (len(kinds) + 5) + "|"]
    for s in SPLITS:
        rs = [r for r in rows if r["split"] == s]
        if not rs:
            continue
        n = {k: sum(r["kind"] == k for r in rs) for k in kinds}
        hard_any = sum(r["hard"] == "1" and r["kind"] in ("hard", "outside") for r in rs)
        rep = sum(int(r["repeat"]) for r in rs)
        gt = sum(r["centre_source"] == "gt" for r in rs)
        lines.append(f"| {s} | {len(rs)} | " + " | ".join(str(n[k]) for k in kinds)
                     + f" | {hard_any} | {rep} | {gt} |")
    text = "\n".join(lines)
    print(text)
    (a.out / "stats.md").write_text(text + "\n")


def stage_preview(a):
    import cv2
    with open(a.out / "manifest.csv", newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["split"] in a.splits]
    rng = np.random.default_rng(a.seed)
    kinds = sorted({r["kind"] for r in rows})
    pick = []
    for k in kinds:
        rs = [r for r in rows if r["kind"] == k]
        pick += [rs[i] for i in rng.choice(len(rs), min(a.n, len(rs)), replace=False)]
    outdir = a.out / "preview"
    outdir.mkdir(exist_ok=True)
    per_page = 8
    for p in range(0, len(pick), per_page):
        tiles = []
        for r in pick[p:p + per_page]:
            img = cv2.imread(str(a.out / r["file"]))
            for k in range(3):
                cv2.line(img, (CROP * k, 0), (CROP * k, CROP), (255, 255, 255), 2)
            for b in filter(None, r["balls"].split(";")):
                x, y, s = map(float, b.split())
                for k in range(3):     # same window position on every frame; the label is for t
                    col = (0, 255, 0) if k == 2 else (0, 160, 255)
                    cv2.circle(img, (int(x) + CROP * k, int(y)), int(max(s, 8)) + 6, col, 2)
            cv2.drawMarker(img, (CROP * 2 + CROP // 2, CROP // 2), (255, 0, 255), cv2.MARKER_CROSS, 20, 1)
            txt = (f"{r['split']} {r['seq']} f{r['frame']} {r['kind']} hard={r['hard']} "
                   f"src={r['centre_source']}{r['centre_gap']} conf={r['main_conf']} n_prev={r['n_prev']}")
            cv2.putText(img, txt, (8, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
            cv2.putText(img, txt, (8, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            tiles.append(cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA))
        path = outdir / f"page_{p // per_page:02d}.jpg"
        cv2.imwrite(str(path), np.concatenate(tiles, axis=0), [cv2.IMWRITE_JPEG_QUALITY, 90])
        print(path)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["build", "stats", "preview"])
    ap.add_argument("--src", type=Path, help="GSR source: sequence dirs or the split zips")
    ap.add_argument("--cache", type=Path, default=Path("data/outputs/ball_temporal/main_cache"))
    ap.add_argument("--out", type=Path, default=Path("data/outputs/ball_temporal/dataset"))
    ap.add_argument("--splits", nargs="+", default=list(SPLITS), choices=SPLITS)
    ap.add_argument("--seqs", nargs="*", default=[], help="only these sequences, e.g. SNGS-024 (smoke tests)")
    ap.add_argument("--workers", type=int, default=min(8, os.cpu_count() or 4))
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--pred_conf", type=float, default=0.15, help="main-detector confidence counted as a sighting")
    ap.add_argument("--jpeg_quality", type=int, default=95)
    ap.add_argument("--n", type=int, default=6, help="preview: samples per kind")
    ap.add_argument("--centre", choices=["last_seen", "predicted"], default="last_seen",
                    help="window centre: the last main sighting (as c5c trained) or the ball tracker's "
                         "prediction (docs/plans/BALL_IMPROVEMENT_PLAN.md §4.3); needs --motion")
    ap.add_argument("--motion", type=Path, default=None, help="camera_motion_cache.py output (--centre predicted)")
    ap.add_argument("--tracker_config", type=Path, default=None,
                    help="tracker_config.json from eval_ball_temporal.py track (its `tracker` section)")
    ap.add_argument("--gap2_repeat", type=int, default=1,
                    help="extra repeat factor for pool hard positives whose window centre is >= 2 frames old")
    a = ap.parse_args()
    if a.stage == "build" and not a.src:
        ap.error("--src is required")
    {"build": stage_build, "stats": stage_stats, "preview": stage_preview}[a.stage](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/ball_eval/eval_ball_temporal.py
"""
eval_ball_temporal.py -- step 4 of docs/path_b/plans/BALL_TEMPORAL_PLAN.md §9: gates 1-3 (plan §8)
for a temporal ball specialist checkpoint (scripts/ball_temporal/train_ball_temporal.py).

Standalone apart from scripts/ball_temporal/ (gsr_io, heatmap_model, build_ball_temporal_dataset,
train_ball_temporal). Needs torch, numpy, opencv-python. Reads the main-detector cache (step 1) and
the crop dataset (step 2).

  crops   gates 1-2: the step 2 crops of a split. Specialist vs main detector on the same crops,
          all positives and hard positives, plus false positives on negative crops.
  system  gate 3: main + specialist on every frame of a split, simulated as the runtime runs it.
  all     both, threshold chosen on `select`, applied unchanged to `test`; writes report.md.

System simulation, per sequence, frame by frame (implementation plan §5.1, temporal plan §6):
  * the main detector's top ball at conf >= --main_conf is the output, and its position is the
    last sighting;
  * otherwise, if the last sighting is at most --max_miss frames old, the specialist runs on
    frames t-2, t-1, t cut by a 640x640 window centred on it (shift-clamped, no camera shift);
    its peak above the threshold is the output;
  * --chain fused (the runtime): a specialist detection becomes the new sighting (window
    re-centres, gap resets). --chain main (as the training crops were built): only main-detector
    sightings place the window.
Scoring: 15 px centre match. recall = hit frames / ball-labelled frames. precision = hits / outputs
on ball-labelled frames (outputs on frames without a ball label are counted separately: those
frames may show an unlabelled ball). Longest miss = longest run of ball-labelled frames not hit.
Threshold (--rule, chosen on `select`, frozen for `test`):
  spec_precision (default, 2026-10-15): the lowest threshold at which >= --spec_precision of the
    specialist's own outputs on ball-labelled frames are on the labelled ball. Independent of the
    main detector's precision, which differs between splits (88.3% select, 85.3% test).
  main_precision (the first rule): the highest recall whose system precision is >= the main
    detector's own on `select`.

    python scripts/ball_eval/eval_ball_temporal.py all --ckpt runs/ball_temporal/c5b/best.pt \
        --src data/datasets/SoccerNetGS --out data/outputs/ball_temporal/eval/c5b
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "scripts" / "ball_temporal"))
sys.path.insert(0, str(Path(__file__).resolve().parent))
from build_ball_temporal_dataset import CROP, clamp_window  # noqa: E402
from gsr_io import SeqReader, decode, discover, parse_labels  # noqa: E402
from heatmap_model import BallHeatmapNet, build_input, decode as decode_hm  # noqa: E402

MATCH_PX = 15.0
FPS = 25
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.05), 2)
GAP_BUCKETS = ((1, 1), (2, 2), (3, 3), (4, 5), (6, 10), (11, 15), (16, 25))


def load_model(ckpt: Path, dev):
    ck = torch.load(ckpt, map_location=dev, weights_only=False)
    m = BallHeatmapNet(ck["channels"], pretrained=False).to(dev).eval()
    m.load_state_dict(ck["model"])
    return m, ck["channels"], ck.get("epoch")


def _near(p, gts) -> bool:
    return any(np.hypot(p[0] - g[0], p[1] - g[1]) <= MATCH_PX for g in gts)


# --------------------------------------------------------------------------- gates 1-2: crops

@torch.no_grad()
def crop_scores(model, channels, data: Path, split: str, dev, workers: int):
    """Specialist peak per crop of a split -> rows with conf, hit (and the manifest's main_conf)."""
    from torch.utils.data import DataLoader
    from train_ball_temporal import TemporalCrops, read_manifest
    rows = read_manifest(data, (split,))
    dl = DataLoader(TemporalCrops(data, rows, False), 32, num_workers=workers)
    for x, pts, _, idx in dl:
        x, pts = x.to(dev), pts.to(dev)
        with torch.autocast("cuda", enabled=dev.type == "cuda"):
            logits, off = model(build_input(x, channels))
        p = decode_hm(logits, off)
        d = torch.cdist(p[:, None, :2], pts[..., :2]).squeeze(1)
        d[pts[..., 2] <= 0] = float("inf")
        hit = (d.min(1).values <= MATCH_PX).cpu().numpy()
        for j, i in enumerate(idx.tolist()):
            rows[i]["spec_conf"] = float(p[j, 2])
            rows[i]["spec_hit"] = bool(hit[j])
    return rows


def crop_report(rows, spec_thr: float, main_conf: float) -> dict:
    """Gates 1-2. Main detector on a crop: a main ball >= main_conf within 15 px of the GT ball
    (positives), or any main ball >= main_conf inside the window (negatives); from the manifest."""
    out = {}
    groups = {"all positives": lambda r: r["kind"] in ("hard", "easy"),
              "hard": lambda r: r["kind"] == "hard", "easy": lambda r: r["kind"] == "easy"}
    for name, f in groups.items():
        rs = [r for r in rows if f(r)]
        out[name] = {"n": len(rs),
                     "main": float(np.mean([float(r["main_conf"]) >= main_conf for r in rs])) if rs else 0.0,
                     "spec": float(np.mean([r["spec_hit"] and r["spec_conf"] >= spec_thr for r in rs])) if rs else 0.0}
    neg = [r for r in rows if r["kind"].startswith("neg")]
    out["negatives fire"] = {"n": len(neg),
                             "main": float(np.mean([float(r["main_conf"]) >= main_conf for r in neg])),
                             "spec": float(np.mean([r["spec_conf"] >= spec_thr for r in neg]))}
    outside = [r for r in rows if r["kind"] == "outside"]
    out["outside fire"] = {"n": len(outside),
                           "main": float(np.mean([float(r["main_conf"]) >= main_conf for r in outside])) if outside else 0.0,
                           "spec": float(np.mean([r["spec_conf"] >= spec_thr for r in outside])) if outside else 0.0}
    return out


def neg_fpr_threshold(rows, fpr: float) -> float:
    neg = np.array([r["spec_conf"] for r in rows if r["kind"].startswith("neg")])
    return float(np.quantile(neg, 1 - fpr))


# --------------------------------------------------------------------------- gate 3: system

class Specialist:
    """Runs the model on one window of frames t-2, t-1, t; memoised by (frame, x0, y0)."""

    def __init__(self, model, channels, dev):
        self.model, self.channels, self.dev = model, channels, dev
        self.memo: dict = {}
        self.calls = 0
        self.ms = 0.0

    @torch.no_grad()
    def __call__(self, key, frames, x0, y0):
        if key in self.memo:
            return self.memo[key]
        x = np.concatenate([f[y0:y0 + CROP, x0:x0 + CROP, ::-1] for f in frames], axis=2)   # BGR->RGB, HxWx9
        x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))[None].to(self.dev)
        t0 = time.perf_counter()
        with torch.autocast("cuda", enabled=self.dev.type == "cuda"):
            logits, off = self.model(build_input(x, self.channels))
        p = decode_hm(logits, off)[0].tolist()
        if self.dev.type == "cuda":
            torch.cuda.synchronize()
        self.ms += (time.perf_counter() - t0) * 1000
        self.calls += 1
        self.memo[key] = (p[0] + x0, p[1] + y0, p[2])
        return self.memo[key]


class SeqData:
    """One sequence: labels, cached main detections, and a small decoded-frame cache."""

    def __init__(self, ref, cache_dir: Path):
        self.ref = ref
        self.reader = SeqReader(ref)
        self.im_dir, self.files, balls, *_rest, (self.W, self.H) = parse_labels(self.reader.labels())
        self.gt = {f: [(b[0], b[1]) for b in bs] for f, bs in balls.items()}
        cache = json.loads((cache_dir / ref.split / f"{ref.name}.json").read_text())["frames"]
        self.dets = {int(k): v["balls"] for k, v in cache.items()}
        self._frames: dict = {}

    def frame(self, f):
        if f not in self._frames:
            if len(self._frames) > 8:
                self._frames.pop(min(self._frames))
            self._frames[f] = decode(self.reader.read(f"{self.im_dir}/{self.files[f]}"))
        return self._frames[f]

    def window_frames(self, t):
        return [self.frame(f if f in self.files else t) for f in (t - 2, t - 1, t)]


def simulate(sd: SeqData, spec: Specialist | None, thr: float, a) -> list[dict]:
    """-> per frame {t, x, y, src, gap} (x None when no output)."""
    out, last = [], None                        # last sighting: (x, y, frame)
    for t in sorted(sd.files):
        m = [b for b in sd.dets.get(t, []) if b[4] >= a.main_conf]
        if m:
            last = (m[0][0], m[0][1], t)
            out.append(dict(t=t, x=m[0][0], y=m[0][1], src="main", gap=None))
            continue
        rec = dict(t=t, x=None, y=None, src="", gap=None)
        if spec is not None and last is not None and t - last[2] <= a.max_miss:
            x0, y0, _ = clamp_window(last[0], last[1], sd.W, sd.H)
            rec["gap"] = t - last[2]
            key = (sd.ref.name, t, x0, y0)
            frames = None if key in spec.memo else sd.window_frames(t)
            px, py, conf = spec(key, frames, x0, y0)
            if conf >= thr:
                rec.update(x=px, y=py, src="spec")
                if a.chain == "fused":
                    last = (px, py, t)
        out.append(rec)
    return out


def score(sd_list, outs) -> dict:
    n_gt = hit = pred_gt = pred_nogt = spec_out = spec_hit = 0
    misses, gap_n, gap_hit = [], defaultdict(int), defaultdict(int)
    for sd, out in zip(sd_list, outs):
        run, longest = 0, 0
        for r in out:
            gts = sd.gt.get(r["t"])
            ok = bool(gts) and r["x"] is not None and _near((r["x"], r["y"]), gts)
            if gts:
                n_gt += 1
                hit += ok
                pred_gt += r["x"] is not None
                if r["src"] == "spec":
                    spec_out += 1
                    spec_hit += ok
                if r["gap"] is not None:
                    b = next((f"{lo}-{hi}" if lo != hi else str(lo) for lo, hi in GAP_BUCKETS if lo <= r["gap"] <= hi), ">25")
                    gap_n[b] += 1
                    gap_hit[b] += ok
                run = 0 if ok else run + 1
                longest = max(longest, run)
            elif r["x"] is not None:
                pred_nogt += 1
        misses.append(longest / FPS)
    return {"gt_frames": n_gt, "recall": hit / max(n_gt, 1), "precision": hit / max(pred_gt, 1),
            "outputs_unlabelled": pred_nogt, "spec_outputs": spec_out,
            "spec_precision": spec_hit / max(spec_out, 1), "spec_added": spec_hit / max(n_gt, 1),
            "longest_miss_mean_s": float(np.mean(misses)), "longest_miss_max_s": float(np.max(misses)),
            "by_gap": {k: (gap_hit[k] / gap_n[k], gap_n[k]) for k in sorted(gap_n, key=lambda s: int(s.split("-")[0].lstrip(">")))}}


def run_system(split, spec, a, thresholds):
    refs = [r for r in discover(a.src) if r.split == split]
    sds = [SeqData(r, a.cache) for r in refs]
    base = score(sds, [simulate(sd, None, 1.0, a) for sd in sds])
    res = {}
    for thr in thresholds:
        t0 = time.time()
        res[float(thr)] = score(sds, [simulate(sd, spec, float(thr), a) for sd in sds])
        print(f"  {split} thr {thr:.2f}: recall {res[float(thr)]['recall']:.2%} precision "
              f"{res[float(thr)]['precision']:.2%} ({time.time() - t0:.0f}s, {spec.calls} specialist calls)", flush=True)
    for sd in sds:
        sd.reader.close()
    return base, res


# --------------------------------------------------------------------------- report

def pct(v):
    return f"{v:.2%}"


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["all", "track"],
                    help="all: gates 1-3 (this file). track: ball tracker + window variants "
                         "(ball_system.py, docs/plans/BALL_IMPROVEMENT_PLAN.md items 2-3)")
    ap.add_argument("--ckpt", type=Path, required=True)
    ap.add_argument("--src", type=Path, default=Path("data/datasets/SoccerNetGS"))
    ap.add_argument("--cache", type=Path, default=Path("data/outputs/ball_temporal/main_cache"))
    ap.add_argument("--data", type=Path, default=Path("data/outputs/ball_temporal/dataset"))
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--main_conf", type=float, default=0.15, help="main-detector output threshold")
    ap.add_argument("--max_miss", type=int, default=15, help="frames since the last sighting the specialist still runs")
    ap.add_argument("--chain", choices=["fused", "main"], default="fused")
    ap.add_argument("--rule", choices=["spec_precision", "main_precision"], default="spec_precision")
    ap.add_argument("--spec_precision", type=float, default=0.90,
                    help="spec_precision rule: required share of specialist outputs on the labelled ball")
    ap.add_argument("--crop_fpr", type=float, default=0.05, help="gates 1-2 threshold: this share of select negatives fire")
    ap.add_argument("--test_sweep", action="store_true",
                    help="also report the test curve at every threshold: a diagnostic, never for choosing")
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--motion", type=Path, default=None, help="track: camera_motion_cache.py output (none = no motion)")
    ap.add_argument("--system_thr", type=float, default=0.70,
                    help="track: the as-wired specialist threshold for the baseline (c5c's sidecar: 0.70)")
    ap.add_argument("--search_passes", type=int, default=2, help="track: coordinate-search passes over the tracker knobs")
    ap.add_argument("--max_seqs", type=int, default=0, help="track: first N sequences per split (smoke tests)")
    a = ap.parse_args()
    dev = torch.device(a.device)
    torch.backends.cudnn.benchmark = True
    a.out.mkdir(parents=True, exist_ok=True)
    model, channels, epoch = load_model(a.ckpt, dev)
    print(f"{a.ckpt} (epoch {epoch}, {channels} channels), chain {a.chain}, max_miss {a.max_miss}", flush=True)
    if a.stage == "track":
        from ball_system import stage_track
        a.chain = "fused"
        stage_track(a, model, channels, dev)
        return

    # gates 1-2
    sel_rows = crop_scores(model, channels, a.data, "select", dev, a.workers)
    crop_thr = neg_fpr_threshold(sel_rows, a.crop_fpr)
    test_rows = crop_scores(model, channels, a.data, "test", dev, a.workers)
    crops = crop_report(test_rows, crop_thr, a.main_conf)
    print(f"crop threshold {crop_thr:.3f} ({a.crop_fpr:.0%} of select negatives fire)", flush=True)

    # gate 3
    spec = Specialist(model, channels, dev)
    sel_base, sel = run_system("select", spec, a, THRESHOLDS)
    if a.rule == "spec_precision":
        ok = [t for t, r in sel.items() if r["spec_precision"] >= a.spec_precision]
        thr = min(ok) if ok else max(sel)
        rule_text = (f"the lowest `select` threshold at which >= {a.spec_precision:.0%} of the specialist's "
                     f"outputs are on the labelled ball")
    else:
        ok = [t for t, r in sel.items() if r["precision"] >= sel_base["precision"]]
        thr = max(ok, key=lambda t: sel[t]["recall"]) if ok else max(sel, key=lambda t: sel[t]["precision"])
        rule_text = f"the highest `select` recall with precision >= the main detector's on `select` ({sel_base['precision']:.2%})"
    print(f"system threshold {thr:.2f} (select: recall {sel[thr]['recall']:.2%}, precision "
          f"{sel[thr]['precision']:.2%}; main alone {sel_base['recall']:.2%} / {sel_base['precision']:.2%})", flush=True)
    spec.calls, spec.ms = 0, 0.0
    test_base, test = run_system("test", spec, a, THRESHOLDS if a.test_sweep else [thr])
    t = test[thr]
    ms = spec.ms / max(spec.calls, 1)      # with --test_sweep, averaged over every pass

    g1 = crops["all positives"]["spec"] > crops["all positives"]["main"]
    g3 = t["recall"] >= test_base["recall"] + 0.10 and t["precision"] >= test_base["precision"]
    L = [f"# Temporal ball specialist: test gates", "",
         f"Checkpoint `{a.ckpt}` (epoch {epoch}, {channels} channels). Main detector cache "
         f"`{a.cache}` (imgsz 1280, output conf >= {a.main_conf}). 15 px centre match. Thresholds "
         f"chosen on `select`, frozen for `test` (15 sequences).", "",
         "## Gates 1-2: crops (step 2 test crops)", "",
         f"Specialist threshold {crop_thr:.3f}: {a.crop_fpr:.0%} of `select` negative crops fire. "
         f"Main detector on a crop: a main ball >= {a.main_conf} within 15 px (positives) or inside the window (negatives).", "",
         "| crops | n | main detector | specialist |", "|---|---|---|---|"]
    for k, v in crops.items():
        L.append(f"| {k} | {v['n']} | {pct(v['main'])} | {pct(v['spec'])} |")
    L += ["", f"Gate 1 (specialist beats main on all test crops): **{'PASS' if g1 else 'FAIL'}**. "
              f"Gate 2 (hard crops): reported above.", "",
          f"## Gate 3: system, every test frame (chain `{a.chain}`, max_miss {a.max_miss})", "",
          f"System threshold {thr:.2f} (rule `{a.rule}`): {rule_text}.", "",
          "| | recall | precision | outputs on unlabelled frames | longest miss mean / max |", "|---|---|---|---|---|",
          f"| main detector | {pct(test_base['recall'])} | {pct(test_base['precision'])} | {test_base['outputs_unlabelled']} | "
          f"{test_base['longest_miss_mean_s']:.1f} s / {test_base['longest_miss_max_s']:.1f} s |",
          f"| main + specialist | {pct(t['recall'])} | {pct(t['precision'])} | {t['outputs_unlabelled']} | "
          f"{t['longest_miss_mean_s']:.1f} s / {t['longest_miss_max_s']:.1f} s |", "",
          f"Recall {(t['recall'] - test_base['recall']) * 100:+.1f} points, precision "
          f"{(t['precision'] - test_base['precision']) * 100:+.1f} points. Specialist outputs on labelled frames: "
          f"{t['spec_outputs']}, {pct(t['spec_precision'])} of them on the labelled ball.", "",
          f"Gate 3 (recall >= main + 10 points, precision >= main): **{'PASS' if g3 else 'FAIL'}**.", "",
          "Specialist recall on labelled frames the main detector missed, by frames since the last sighting:", "",
          "| gap | frames | recall |", "|---|---|---|"]
    for k, (r, n) in t["by_gap"].items():
        L.append(f"| {k} | {n} | {pct(r)} |")
    L += ["", f"`select` sweep (main alone: recall {pct(sel_base['recall'])}, precision {pct(sel_base['precision'])}):", "",
          "| threshold | recall | precision | specialist outputs on the ball |", "|---|---|---|---|"]
    for k, r in sel.items():
        L.append(f"| {k:.2f} | {pct(r['recall'])} | {pct(r['precision'])} | {pct(r['spec_precision'])} |")
    if a.test_sweep:
        L += ["", "`test` sweep, **diagnostic only** (the gate uses the `select` threshold above):", "",
              "| threshold | recall | precision |", "|---|---|---|"]
        for k, r in test.items():
            L.append(f"| {k:.2f} | {pct(r['recall'])} | {pct(r['precision'])} |")
    L += ["", f"Specialist PyTorch FP16 forward + decode, batch 1, this machine: {ms:.1f} ms per call "
              f"({spec.calls} calls on test). Not the gate 4 figure (that is TensorRT, step 5)."]
    text = "\n".join(L)
    print("\n" + text)
    (a.out / "report.md").write_text(text + "\n", encoding="utf-8")
    (a.out / "results.json").write_text(json.dumps(
        {"ckpt": str(a.ckpt), "epoch": epoch, "channels": channels, "chain": a.chain, "max_miss": a.max_miss,
         "rule": a.rule, "spec_precision_target": a.spec_precision,
         "crop_thr": crop_thr, "crops": crops, "system_thr": thr, "select_main": sel_base, "select": sel,
         "test_main": test_base, "test": t, "ms_per_call": ms, "gate1": g1, "gate3": g3}, indent=1, default=str))
    with open(a.out / "test_crops.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(test_rows[0]))
        w.writeheader()
        w.writerows(test_rows)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/gsr_pipeline/ball_gate.py
"""
ball_gate.py -- plausibility check on the main detector's ball candidates
(docs/plans/PIPELINE_IMPROVEMENT_PLAN_1.md §3.1, Phase 1 task 1).

Without the gate, BallFuser accepts the detector's top ball unconditionally. Phase 0 found that
a wrong detector ball is the largest miss cause (10.2% of test ball frames), and that a wrong
sighting then misplaces the specialist window (another 7.7%). The gate accepts the first
candidate (highest confidence first) that is plausible given the last sighting:

  * there is no last sighting, or it is more than `max_miss` frames old  -> accept the top one;
  * conf >= conf_hi                                                      -> accept;
  * distance to the last sighting <= r0 + speed * gap                    -> accept.

If no candidate passes, the frame is treated as a detector miss (the specialist runs at the
last sighting). After `max_rejects` consecutive rejected frames the top candidate is accepted
anyway, so a real fast change of position (long pass, cut) cannot lock the ball out.

Pure Python, no dependencies: imported by gsr_pipeline/ball_specialist.py (runtime) and
scripts/pipeline_improvement_1/phase1/phase1_sweep.py (evaluation), so both run the same rule.
"""

from __future__ import annotations

import math
from dataclasses import dataclass, asdict
from typing import Optional, Sequence


@dataclass(frozen=True)
class GateParams:
    enabled: bool = False      # False = accept the top candidate (the pre-Phase-1 behaviour)
    r0: float = 30.0           # px allowed at any gap (detector jitter, box-centre noise)
    speed: float = 60.0        # px per frame of gap (ball p99 ~53 px at 1 frame, 1920x1080)
    conf_hi: float = 1.01      # a candidate at or above this is accepted regardless (>1 = never)
    max_rejects: int = 10      # consecutive rejected frames before the top candidate is forced

    def as_dict(self) -> dict:
        return asdict(self)

    def tag(self) -> str:
        if not self.enabled:
            return "off"
        return f"r{self.r0:g}_v{self.speed:g}_c{self.conf_hi:g}_k{self.max_rejects}"


class BallGate:
    """Per-stream state (the consecutive-reject counter). Create one per stream or call reset."""

    def __init__(self, params: Optional[GateParams] = None, max_miss: int = 15) -> None:
        self.p = params or GateParams()
        self.max_miss = max_miss
        self.reset()

    def reset(self) -> None:
        self.rejects = 0

    def pick(self, cands: Sequence[tuple], t: int, last: Optional[tuple]) -> Optional[int]:
        """cands: (x, y, conf, ...) sorted by conf, highest first, already at the output conf.
        last: (x, y, frame) of the last sighting or None. Returns the accepted index or None."""
        if not cands:
            return None
        p = self.p
        if not p.enabled or last is None or t - last[2] > self.max_miss:
            self.rejects = 0
            return 0
        allowed = p.r0 + p.speed * max(t - last[2], 1)
        for i, c in enumerate(cands):
            if c[2] >= p.conf_hi or math.hypot(c[0] - last[0], c[1] - last[1]) <= allowed:
                self.rejects = 0
                return i
        self.rejects += 1
        if self.rejects > p.max_rejects:
            self.rejects = 0
            return 0
        return None

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/pipeline_improvement_1/phase1/phase1_sweep.py
"""
phase1_sweep.py -- Phase 1 of docs/plans/PIPELINE_IMPROVEMENT_PLAN_1.md: the detector-ball
plausibility gate (gsr_pipeline/ball_gate.py, task 1) and the c5c threshold sweep (task 2).

Replays main + c5c on cached main detections, exactly as BallFuser.update runs it (fused chain;
with the gate: first plausible candidate, none plausible = detector miss). The specialist is
memoised per window, so configurations that share windows share GPU work.

Search, on `select` only:
  A. gate grid at threshold 0.70 (plus gate off);
  B. thresholds 0.40-0.85 for gate off and the best --top_gates gates of A.
  Rule: highest select recall with select precision >= gate-off@0.70 precision - --prec_drop.
Then, unchanged on `test`: baseline, gate only, threshold only, gate + threshold, each also
without --leak_seqs (SNGS-050 overlaps a `pool` clip c5c trained on). A candidate is kept only
if test recall gains >= --min_gain and test precision drops <= --prec_drop (the plan's kill
criterion); of the kept ones the highest test recall is chosen.

Outputs (--out): report_phase1.md, phase1.json, sweep.csv, phase1_log.txt, and
frames_<chosen>/frames.jsonl in diagnose_ball_misses.py's format (feed it to
scripts/pipeline_improvement_1/phase0/phase0_report.py --diag for the new miss buckets).

    python scripts/pipeline_improvement_1/phase1/phase1_sweep.py --src data/datasets/SoccerNetGS \
        --cache data/outputs/pipeline_improvement_1/phase0/main_cache \
        --ckpt runs/ball_temporal/c5c/best.pt --out data/outputs/pipeline_improvement_1/phase1
"""

from __future__ import annotations

import argparse
import csv
import itertools
import json
import sys
import time
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve()
ROOT = HERE.parents[3]
sys.path.insert(0, str(ROOT / "gsr_pipeline"))          # ball_gate.py alone, without the package
sys.path.insert(0, str(ROOT / "scripts" / "ball_temporal"))
sys.path.insert(0, str(ROOT / "scripts" / "ball_eval"))
from ball_gate import BallGate, GateParams  # noqa: E402
from eval_ball_temporal import SeqData, Specialist, clamp_window, load_model  # noqa: E402
from gsr_io import discover  # noqa: E402

TOLS = (5.0, 10.0, 15.0)
MATCH = 15.0
BASE_THR = 0.70
THRESHOLDS = [round(float(x), 2) for x in np.arange(0.40, 0.851, 0.05)]
GRID = dict(r0=[30.0], speed=[30.0, 50.0, 80.0], conf_hi=[0.6, 0.8, 1.01], max_rejects=[5, 15])


def gate_grid():
    return [GateParams(enabled=True, **dict(zip(GRID, v))) for v in itertools.product(*GRID.values())]


def simulate(sd: SeqData, spec: Specialist, thr: float, gp: GateParams, a) -> list[dict]:
    """BallFuser.update (fused chain) with the gate. Per frame: t, out, src, win, sconf, gap, rej."""
    gate = BallGate(gp, a.max_miss)
    out, last = [], None
    for t in sorted(sd.files)[: a.max_frames or None]:
        cands = sorted((b for b in sd.dets.get(t, []) if b[4] >= a.main_conf), key=lambda b: -b[4])
        rec = dict(t=t, out=None, src="", win=None, sconf=None, gap=None, rej=None)
        i = gate.pick([(b[0], b[1], b[4]) for b in cands], t, last)
        if i is not None:
            last = (cands[i][0], cands[i][1], t)
            rec.update(out=(cands[i][0], cands[i][1]), src="main")
            out.append(rec)
            continue
        if cands:
            rec["rej"] = (cands[0][0], cands[0][1])
        if last is not None and t - last[2] <= a.max_miss:
            x0, y0, _ = clamp_window(last[0], last[1], sd.W, sd.H)
            rec.update(gap=t - last[2], win=(int(x0), int(y0)))
            key = (sd.ref.name, t, x0, y0)
            px, py, conf = spec(key, None if key in spec.memo else sd.window_frames(t), x0, y0)
            rec["sconf"] = float(conf)
            if conf >= thr:
                rec.update(out=(px, py), src="spec")
                last = (px, py, t)
        out.append(rec)
    return out


def _d(p, gts):
    return min((float(np.hypot(p[0] - g[0], p[1] - g[1])) for g in gts), default=np.inf) if p else np.inf


def score(sds, outs, exclude=()) -> dict:
    n = pred = unl = rej = rej_good = calls = 0
    hit = {t: 0 for t in TOLS}
    longest = []
    for sd, out in zip(sds, outs):
        if sd.ref.name in exclude:
            continue
        run = lg = 0
        for r in out:
            gts = sd.gt.get(r["t"])
            calls += r["win"] is not None
            if r["rej"] is not None:
                rej += 1
                rej_good += bool(gts) and _d(r["rej"], gts) <= MATCH
            if not gts:
                unl += r["out"] is not None
                continue
            n += 1
            d = _d(r["out"], gts)
            pred += r["out"] is not None
            for t in TOLS:
                hit[t] += d <= t
            run = 0 if d <= MATCH else run + 1
            lg = max(lg, run)
        longest.append(lg)
    return {"ball_frames": n, "recall": hit[15.0] / max(n, 1), "recall5": hit[5.0] / max(n, 1),
            "recall10": hit[10.0] / max(n, 1), "precision": hit[15.0] / max(pred, 1),
            "unlabelled_outputs": unl, "spec_calls": calls, "gate_rejects": rej,
            "gate_rejects_correct": rej_good, "longest_miss_frames": max(longest, default=0)}


def run_configs(sds, spec, configs, a, log, exclude=()):
    rows = []
    for gp, thr in configs:
        t0, c0 = time.time(), spec.calls
        outs = [simulate(sd, spec, thr, gp, a) for sd in sds]
        s = score(sds, outs)
        row = dict(gate=gp.tag(), thr=thr, **s, new_calls=spec.calls - c0, secs=round(time.time() - t0, 1),
                   params=gp.as_dict(), _outs=outs)
        if exclude:
            ex = score(sds, outs, exclude)
            row.update(recall_ex=ex["recall"], precision_ex=ex["precision"])
        rows.append(row)
        log(f"  {gp.tag():>22} thr {thr:.2f}: recall {s['recall']:.2%}  precision {s['precision']:.2%}  "
            f"rejects {s['gate_rejects']} ({s['gate_rejects_correct']} correct)  +{row['new_calls']} calls  {row['secs']}s")
    return rows


def best(rows, floor):
    ok = [r for r in rows if r["precision"] >= floor]
    return max(ok, key=lambda r: (r["recall"], r["precision"])) if ok else None


def write_frames(fdir: Path, split, sds, outs):
    with open(fdir / "frames.jsonl", "a") as f:
        for sd, out in zip(sds, outs):
            for r in out:
                gts = sd.gt.get(r["t"], [])
                hit = bool(gts) and _d(r["out"], gts) <= MATCH
                f.write(json.dumps(dict(split=split, seq=sd.ref.name, t=r["t"], gt=[list(g) for g in gts],
                                        out=list(r["out"]) if r["out"] else None, src=r["src"], win=r["win"],
                                        sconf=r["sconf"], gap=r["gap"], hit=float(hit))) + "\n")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--src", type=Path, required=True, help="GSR root (gsr_io.discover)")
    ap.add_argument("--cache", type=Path, required=True, help="main-detector cache (Phase 0 main_cache/)")
    ap.add_argument("--ckpt", type=Path, required=True, help="c5c best.pt")
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--main_conf", type=float, default=0.15)
    ap.add_argument("--max_miss", type=int, default=15)
    ap.add_argument("--top_gates", type=int, default=3)
    ap.add_argument("--prec_drop", type=float, default=0.01, help="precision guardrail (fraction)")
    ap.add_argument("--min_gain", type=float, default=0.005, help="kill criterion: minimum test recall gain")
    ap.add_argument("--leak_seqs", default="SNGS-050", help="comma list, test also reported without them")
    ap.add_argument("--max_seqs", type=int, default=0, help="smoke: first N sequences per split")
    ap.add_argument("--max_frames", type=int, default=0, help="smoke: first N frames per sequence")
    a = ap.parse_args()
    a.out.mkdir(parents=True, exist_ok=True)
    logf = open(a.out / "phase1_log.txt", "w")

    def log(msg):
        print(msg, flush=True)
        logf.write(msg + "\n")
        logf.flush()

    import torch
    dev = torch.device(a.device if torch.cuda.is_available() else "cpu")
    model, channels, epoch = load_model(a.ckpt, dev)
    spec = Specialist(model, channels, dev)
    log(f"c5c {a.ckpt} (epoch {epoch}) on {dev}")
    refs = discover(a.src)
    leak = tuple(s for s in a.leak_seqs.split(",") if s)

    def load(split):
        return [SeqData(r, a.cache) for r in [r for r in refs if r.split == split][: a.max_seqs or None]]

    sel = load("select")
    log(f"select: {len(sel)} sequences\n[A] gate grid at threshold {BASE_THR}")
    rows_a = run_configs(sel, spec, [(GateParams(), BASE_THR)] + [(g, BASE_THR) for g in gate_grid()], a, log)
    base_sel = rows_a[0]
    floor = base_sel["precision"] - a.prec_drop
    gp_of = {r["gate"]: GateParams(**r["params"]) for r in rows_a}
    top = [r["gate"] for r in sorted((r for r in rows_a[1:] if r["precision"] >= floor), key=lambda r: -r["recall"])[: a.top_gates]]
    log(f"[B] thresholds {THRESHOLDS} for gate off + {top}")
    rows_b = run_configs(sel, spec, [(gp_of[g], t) for g in ["off"] + top for t in THRESHOLDS if t != BASE_THR], a, log)
    allrows = rows_a + rows_b
    picks = {"baseline": base_sel,
             "threshold_only": best([r for r in allrows if r["gate"] == "off"], floor),
             "gate_only": best([r for r in allrows if r["gate"] != "off" and r["thr"] == BASE_THR], floor),
             "gate_threshold": best(allrows, floor)}
    picks = {k: v for k, v in picks.items() if v is not None}
    for sd in sel:
        sd.reader.close()

    test = load("test")
    log(f"test: {len(test)} sequences; frozen picks: " + ", ".join(f"{k}={v['gate']}@{v['thr']}" for k, v in picks.items()))
    test_of = dict(zip(picks, run_configs(test, spec, [(gp_of[v["gate"]], v["thr"]) for v in picks.values()],
                                          a, log, exclude=leak)))
    bt = test_of["baseline"]
    verdict = {k: ("keep" if r["recall"] - bt["recall"] >= a.min_gain and bt["precision"] - r["precision"] <= a.prec_drop
                   else "revert") for k, r in test_of.items() if k != "baseline"}
    kept = [k for k, v in verdict.items() if v == "keep"]
    chosen = max(kept, key=lambda k: test_of[k]["recall"]) if kept else "baseline"

    fdir = a.out / f"frames_{chosen}"
    fdir.mkdir(parents=True, exist_ok=True)
    (fdir / "frames.jsonl").unlink(missing_ok=True)
    sel = load("select")
    write_frames(fdir, "select", sel, [simulate(sd, spec, picks[chosen]["thr"], gp_of[picks[chosen]["gate"]], a) for sd in sel])
    write_frames(fdir, "test", test, test_of[chosen]["_outs"])
    for sd in sel + test:
        sd.reader.close()

    cols = ["split", "gate", "thr", "recall", "recall5", "recall10", "precision", "recall_ex", "precision_ex",
            "unlabelled_outputs", "spec_calls", "gate_rejects", "gate_rejects_correct", "longest_miss_frames",
            "new_calls", "secs"]
    with open(a.out / "sweep.csv", "w", newline="") as f:
        w = csv.DictWriter(f, cols, extrasaction="ignore")
        w.writeheader()
        for r in allrows:
            w.writerow(dict(r, split="select"))
        for k, r in test_of.items():
            w.writerow(dict(r, split=f"test:{k}"))

    strip = lambda r: {k: v for k, v in r.items() if k != "_outs"}  # noqa: E731
    json.dump({"select": {k: strip(v) for k, v in picks.items()}, "test": {k: strip(v) for k, v in test_of.items()},
               "verdict": verdict, "chosen": chosen, "chosen_params": gp_of[picks[chosen]["gate"]].as_dict(),
               "chosen_threshold": picks[chosen]["thr"], "leak_seqs": leak, "select_precision_floor": floor,
               "spec_calls_total": spec.calls, "spec_ms_mean": spec.ms / max(spec.calls, 1)},
              open(a.out / "phase1.json", "w"), indent=1)

    pc = lambda x: f"{x:.2%}"  # noqa: E731
    L = ["# Phase 1: plausibility gate + c5c threshold (PIPELINE_IMPROVEMENT_PLAN_1.md)", "",
         f"Chosen on `select` (precision floor {pc(floor)} = gate off @ {BASE_THR} minus {a.prec_drop:.0%}), frozen "
         f"for `test`. Kill criterion: test recall gain >= {a.min_gain:.1%} and precision drop <= {a.prec_drop:.0%}.", "",
         "## Test", "",
         f"| config | gate | thr | recall @15 | @10 | @5 | precision | recall w/o {','.join(leak)} | rejects (correct) | spec calls | longest miss | verdict |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in test_of.items():
        L.append(f"| {k} | `{r['gate']}` | {r['thr']} | {pc(r['recall'])} | {pc(r['recall10'])} | {pc(r['recall5'])} | "
                 f"{pc(r['precision'])} | {pc(r['recall_ex'])} | {r['gate_rejects']} ({r['gate_rejects_correct']}) | "
                 f"{r['spec_calls']} | {r['longest_miss_frames']} | {verdict.get(k, '—')} |")
    L += ["", f"**Chosen: `{chosen}`.** Replay: `frames_{chosen}/frames.jsonl`; buckets: `buckets_{chosen}/`.", "",
          "\"rejects (correct)\": frames where the gate rejected the detector's top ball, and how many of those "
          "were actually on the ball (the gate's cost).", "",
          "## Select search (top 25 by recall)", "", "| gate | thr | recall | precision | rejects (correct) |",
          "|---|---|---|---|---|"]
    for r in sorted(allrows, key=lambda r: -r["recall"])[:25]:
        L.append(f"| `{r['gate']}` | {r['thr']} | {pc(r['recall'])} | {pc(r['precision'])} | "
                 f"{r['gate_rejects']} ({r['gate_rejects_correct']}) |")
    L += ["", f"Full grid: `sweep.csv`. Specialist calls run: {spec.calls}, mean {spec.ms / max(spec.calls, 1):.1f} ms."]
    (a.out / "report_phase1.md").write_text("\n".join(L) + "\n", encoding="utf-8")
    log(f"chosen: {chosen}; wrote {a.out / 'report_phase1.md'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/pipeline_improvement_1/phase0/phase0_report.py
"""
phase0_report.py -- Phase 0 of docs/plans/PIPELINE_IMPROVEMENT_PLAN_1.md: the reproducible
baseline, the miss breakdown and the ceilings, from one replay of main + c5c as wired.

CPU only. Runs after scripts/ball_eval/diagnose_ball_misses.py (which replays
eval_ball_temporal's fused chain = BallFuser.update, on GPU) and reads its frames.jsonl, plus the
GSR labels for ball sizes, person boxes and split metadata.

    python scripts/pipeline_improvement_1/phase0/phase0_report.py \
        --src <GSR root> --diag <diagnose out> --out <phase0 out> [--frames_per_hash 25]

Outputs (<out>/):
  report_phase0.md     everything below, human-readable
  phase0.json          the same numbers, machine-readable
  buckets.csv          one row per miss: bucket, cross-flags, size, gap
  split_integrity.csv  one row per sequence: split, game_id, clip range, flags
  near_duplicates.csv  only if any near-duplicate frames were found

What it measures, per split (select, test):
  1. stages at 5/10/15 px: main alone, the specialist on the calls it is given (with crop
     containment), and the fused output. recall = hits / ball-labelled frames; precision = hits /
     outputs on ball-labelled frames (eval_ball_temporal's definitions).
  2. recall by ball size (max side, px): <10, 10-15, 15-20, 20+.
  3. temporal continuity: longest miss, miss-run p50/p95 (= recovery time), outputs on unlabelled frames.
  4. the miss breakdown (plan §5 Phase 0). Every 15 px miss gets exactly one bucket:
       B1 long_gap          no main ball, last sighting > 15 frames old: nothing searched
       B2 no_prior          no main ball and no sighting yet in the sequence
       B3 false_sighting    specialist ran, but the sighting that placed its window was wrong
       B4 outside_window    specialist ran from a correct sighting, ball outside the 640 window
       B5 below_threshold   ball inside the window, specialist peak < threshold
       B6 spec_wrong_place  specialist output >= threshold but > 15 px from the ball
       B7 main_false        main detector output on this frame, > 15 px from the ball
       B0 replay_check      recent sighting yet no specialist call: must be 0 unless --no_spec
     Cross-flags (not buckets): occluded_proxy (centre inside a person box +10%; the labels have
     no per-ball visibility, so this stands in for "hidden"), small (< 10 px).
  5. ceilings: recall if a bucket were fully fixed, and the occluded-proxy share.
  6. split integrity: sequence-name clashes, same game + overlapping clip time across splits, and
     near-duplicate frames (64-bit dHash, Hamming <= 4) between {select, test} and {train, pool}.
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[3]
sys.path.insert(0, str(ROOT / "scripts" / "ball_temporal"))
from gsr_io import SeqReader, decode, discover, parse_labels  # noqa: E402

CROP = 640
FPS = 25
TOLS = (5.0, 10.0, 15.0)
MATCH = 15.0
SIZE_BUCKETS = ((0, 10, "<10"), (10, 15, "10-15"), (15, 20, "15-20"), (20, 1e9, "20+"))
BUCKETS = {
    "B0": ("replay_check", "should be 0 in a normal replay: investigate if not"),
    "B1": ("long_gap", "Phase 4 tracker / reacquisition"),
    "B2": ("no_prior", "Phase 2 detector"),
    "B3": ("false_sighting", "Phase 1 plausibility check (§3.1)"),
    "B4": ("outside_window", "Phase 1 window shift (§3.2)"),
    "B5": ("below_threshold", "Phase 3 specialist / threshold"),
    "B6": ("spec_wrong_place", "Phase 3 specialist localisation"),
    "B7": ("main_false", "Phase 1 plausibility check / Phase 2 detector"),
}
COUNT_KEYS = ("n_gt", "unlabelled_outputs", "spec_calls", "spec_calls_gt", "spec_contained", "spec_hits",
              "spec_hits_contained")


def new_agg():
    return {**{k: 0 for k in COUNT_KEYS},
            "main": {t: [0, 0] for t in TOLS}, "fused": {t: [0, 0] for t in TOLS},
            "size": defaultdict(lambda: [0, 0, 0])}   # size bucket -> [frames, main hits, fused hits]


def nearest(o, gts):
    if o is None:
        return np.inf
    return min((float(np.hypot(o[0] - g[0], o[1] - g[1])) for g in gts), default=np.inf)


def load_frames(diag: Path):
    by_seq = defaultdict(list)
    with open(diag / "frames.jsonl") as f:
        for line in f:
            r = json.loads(line)
            by_seq[(r["split"], r["seq"])].append(r)
    for v in by_seq.values():
        v.sort(key=lambda r: r["t"])
    return by_seq


def runs_of(mask):
    out, cur = [], 0
    for m in mask:
        if m:
            cur += 1
        elif cur:
            out.append(cur)
            cur = 0
    if cur:
        out.append(cur)
    return out


def analyse_seq(rows, labels, max_miss):
    """One sequence of the fused replay -> counts, per-miss rows, miss-run lengths."""
    _im, _files, balls, persons, _lines, _wh = labels
    st = new_agg()
    misses, miss_mask = [], []
    last_t, last_ok = None, None   # the fused chain's last sighting, and whether it was on the ball
    for r in rows:
        t, o, src = r["t"], r["out"], r["src"]
        gts = [tuple(g) for g in r["gt"]]
        d = nearest(o, gts)
        hit = bool(gts) and d <= MATCH
        lab = balls[t][0] if balls.get(t) else None
        size = max(lab[2], lab[3]) if lab else 0.0

        if gts:
            st["n_gt"] += 1
            for tol in TOLS:
                if src == "main":
                    st["main"][tol][1] += 1
                    st["main"][tol][0] += d <= tol
                if o is not None:
                    st["fused"][tol][1] += 1
                    st["fused"][tol][0] += d <= tol
            sb = next(n for lo, hi, n in SIZE_BUCKETS if lo <= size < hi)
            s = st["size"][sb]
            s[0] += 1
            s[1] += src == "main" and hit
            s[2] += hit
        elif o is not None:
            st["unlabelled_outputs"] += 1

        ran = r["win"] is not None
        contained = None
        if ran:
            st["spec_calls"] += 1
            if gts:
                x0, y0 = r["win"]
                contained = x0 <= gts[0][0] < x0 + CROP and y0 <= gts[0][1] < y0 + CROP
                spec_hit = src == "spec" and hit
                st["spec_calls_gt"] += 1
                st["spec_contained"] += contained
                st["spec_hits"] += spec_hit
                st["spec_hits_contained"] += spec_hit and contained

        if gts:
            miss_mask.append(not hit)
        if gts and not hit:
            g = gts[0]
            if src == "main":
                b = "B7"
            elif ran:
                b = "B3" if last_ok is False else "B4" if not contained else "B6" if src == "spec" else "B5"
            elif last_t is None:
                b = "B2"
            elif t - last_t > max_miss:
                b = "B1"
            else:
                b = "B0"   # recent sighting but no specialist call: --no_spec, or replay and report disagree
            occ = any(abs(g[0] - p[0]) <= 0.55 * p[2] and abs(g[1] - p[1]) <= 0.55 * p[3] for p in persons.get(t, []))
            misses.append(dict(seq=r["seq"], t=t, bucket=b, name=BUCKETS[b][0], occluded_proxy=int(occ),
                               small=int(0 < size < 10), size=round(size, 1),
                               gap=(t - last_t) if last_t is not None else "",
                               sconf=round(r["sconf"], 3) if r["sconf"] is not None else ""))
        if o is not None:
            last_t, last_ok = t, hit
    return st, misses, runs_of(miss_mask)


def pct(a, b):
    return a / b if b else 0.0


def split_block(split, agg, misses, runs):
    n = agg["n_gt"]
    L = [f"## `{split}`: {agg['seqs']} sequences, {n} ball-labelled frames", "",
         "### Stages", "",
         "| stage | recall @5 px | @10 px | @15 px | precision @15 px |", "|---|---|---|---|---|"]
    out = {"sequences": agg["seqs"], "ball_frames": n}
    for name, label in (("main", "main detector alone"), ("fused", "main + c5c (fused, as wired)")):
        rec = [pct(agg[name][t][0], n) for t in TOLS]
        prec = pct(agg[name][MATCH][0], agg[name][MATCH][1])
        out[name] = {"recall": {f"{t:g}": v for t, v in zip(TOLS, rec)}, "precision_15": prec}
        L.append(f"| {label} | {rec[0]:.1%} | {rec[1]:.1%} | {rec[2]:.1%} | {prec:.1%} |")
    sc = agg["spec_calls_gt"]
    spec = {"calls": agg["spec_calls"], "calls_on_ball_frames": sc,
            "containment": pct(agg["spec_contained"], sc), "recall_on_calls": pct(agg["spec_hits"], sc),
            "recall_when_contained": pct(agg["spec_hits_contained"], agg["spec_contained"])}
    out["specialist"] = spec
    out["unlabelled_outputs"] = agg["unlabelled_outputs"]
    L += ["", f"Specialist: {spec['calls']} calls, {sc} on ball-labelled frames. Ball inside the window "
              f"(crop containment): {spec['containment']:.1%}. Recall on its calls: {spec['recall_on_calls']:.1%}; "
              f"when the ball was inside the window: {spec['recall_when_contained']:.1%}. "
              f"Outputs on frames with no ball label: {agg['unlabelled_outputs']} (may be unlabelled balls).", "",
          "### Recall by ball size (max side, px; 15 px match)", "",
          "| size | frames | main | fused |", "|---|---|---|---|"]
    out["by_size"] = {}
    for _lo, _hi, nm in SIZE_BUCKETS:
        s = agg["size"].get(nm, [0, 0, 0])
        out["by_size"][nm] = {"frames": s[0], "main": pct(s[1], s[0]), "fused": pct(s[2], s[0])}
        L.append(f"| {nm} | {s[0]} | {pct(s[1], s[0]):.1%} | {pct(s[2], s[0]):.1%} |")
    r = np.array(runs) if runs else np.array([0])
    cont = {"miss_runs": len(runs), "longest_frames": int(r.max()), "longest_s": float(r.max()) / FPS,
            "run_p50": float(np.percentile(r, 50)), "run_p95": float(np.percentile(r, 95))}
    out["continuity"] = cont
    L += ["", "### Temporal continuity", "",
          f"{cont['miss_runs']} miss runs. Longest: {cont['longest_frames']} frames ({cont['longest_s']:.1f} s). "
          f"Run length (frames to recover): p50 {cont['run_p50']:.0f}, p95 {cont['run_p95']:.0f}.", "",
          "### Miss breakdown (15 px)", "",
          "| bucket | misses | share of misses | share of ball frames | occluded-proxy | < 10 px | addressed by |",
          "|---|---|---|---|---|---|---|"]
    c = Counter(m["bucket"] for m in misses)
    occ = Counter(m["bucket"] for m in misses if m["occluded_proxy"])
    sml = Counter(m["bucket"] for m in misses if m["small"])
    nm_ = max(len(misses), 1)
    out["buckets"] = {}
    for b, (name, by) in BUCKETS.items():
        out["buckets"][b] = {"name": name, "misses": c[b], "share_of_misses": c[b] / nm_,
                             "share_of_ball_frames": pct(c[b], n), "occluded_proxy": occ[b], "small": sml[b]}
        L.append(f"| {b} {name} | {c[b]} | {c[b] / nm_:.1%} | {pct(c[b], n):.1%} | {occ[b]} | {sml[b]} | {by} |")
    occ_all = sum(m["occluded_proxy"] for m in misses)
    L.append(f"| **total** | {len(misses)} | 100% | {pct(len(misses), n):.1%} | {occ_all} | "
             f"{sum(m['small'] for m in misses)} | |")
    fused = out["fused"]["recall"]["15"]
    ceil = {b: fused + pct(c[b], n) for b in BUCKETS}
    ceil["phase1_B3_B4_B7"] = fused + pct(c["B3"] + c["B4"] + c["B7"], n)
    ceil["all_but_occluded_proxy"] = 1.0 - pct(occ_all, n)
    out["ceilings"] = ceil
    out["occluded_proxy_share"] = pct(occ_all, n)
    L += ["", "### Ceilings (recall @15 px if that bucket were fully fixed)", "",
          "| fix | recall |", "|---|---|", f"| now | {fused:.1%} |"]
    for b, (name, _) in BUCKETS.items():
        L.append(f"| {b} {name} | {ceil[b]:.1%} |")
    L += [f"| B3 + B4 + B7 (Phase 1, no new model) | {ceil['phase1_B3_B4_B7']:.1%} |",
          f"| every miss except the occluded-proxy ones | {ceil['all_but_occluded_proxy']:.1%} |", "",
          f"Occluded-proxy misses are {pct(occ_all, n):.1%} of ball frames. This is a proxy for \"hidden\", "
          "not a visibility label: some are visible balls in front of a player. The plan's ~10% rule (§6) "
          "should be applied only after the audit sheets confirm how many are truly hidden.", ""]
    return L, out


def dhash(img, size=8):
    import cv2
    g = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (size + 1, size), interpolation=cv2.INTER_AREA)
    v = 0
    for bit in (g[:, 1:] > g[:, :-1]).flatten():
        v = (v << 1) | int(bit)
    return v


def split_integrity(refs, every):
    rows, hashes = [], defaultdict(list)
    for ref in refs:
        rd = SeqReader(ref)
        try:
            d = rd.labels()
            info = d.get("info", {})
            im_dir, files, *_ = parse_labels(d)
            rows.append(dict(seq=ref.name, src_split=ref.src_split, split=ref.split,
                             game_id=str(info.get("game_id", "")),
                             clip_start=int(info.get("clip_start") or -1), clip_stop=int(info.get("clip_stop") or -1),
                             game_time_start=info.get("game_time_start", ""), flags=""))
            if every:
                for f in sorted(files)[::every]:
                    try:
                        hashes[ref.split].append((ref.name, f, dhash(decode(rd.read(f"{im_dir}/{files[f]}")))))
                    except Exception:
                        pass
        finally:
            rd.close()
    findings = []
    for h in (r for r in rows if r["split"] in ("select", "test")):
        overl = [o for o in rows if o is not h and o["game_id"] and o["game_id"] == h["game_id"]
                 and o["src_split"] == h["src_split"] and o["split"] != h["split"]
                 and o["clip_start"] < h["clip_stop"] and h["clip_start"] < o["clip_stop"]]
        if overl:
            h["flags"] += "clip_overlap:" + "|".join(o["seq"] for o in overl) + ";"
            findings.append(f"{h['seq']} ({h['split']}) overlaps in time with "
                            + ", ".join(f"{o['seq']} ({o['split']})" for o in overl))
    dup = []
    ref_h = [x for s in ("train", "pool") for x in hashes.get(s, [])]
    if ref_h:
        ref_v = np.array([v for _, _, v in ref_h], dtype=np.uint64)
        for s in ("select", "test"):
            for n, f, v in hashes.get(s, []):
                x = np.bitwise_xor(ref_v, np.uint64(v))
                ham = np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(1)
                j = int(ham.argmin())
                if ham[j] <= 4:
                    dup.append(dict(split=s, seq=n, frame=f, match_seq=ref_h[j][0], match_frame=ref_h[j][1],
                                    hamming=int(ham[j])))
    names = {s: sorted(r["seq"] for r in rows if r["split"] == s) for s in ("train", "pool", "select", "test")}
    clash = sorted(set(names["test"]) & (set(names["train"]) | set(names["pool"]) | set(names["select"])))
    return rows, findings, dup, names, clash


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--src", type=Path, required=True)
    ap.add_argument("--diag", type=Path, required=True, help="diagnose_ball_misses.py --out")
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--splits", nargs="+", default=["select", "test"])
    ap.add_argument("--max_miss", type=int, default=15)
    ap.add_argument("--frames_per_hash", type=int, default=25, help="hash every Nth frame for near-duplicates (0 = off)")
    a = ap.parse_args()
    a.out.mkdir(parents=True, exist_ok=True)

    refs = discover(a.src)
    by_name = {r.name: r for r in refs}
    frames = load_frames(a.diag)
    dpath = a.diag / "diagnosis.json"
    diag_meta = json.loads(dpath.read_text()) if dpath.exists() else {}
    report = ["# Phase 0 baseline (PIPELINE_IMPROVEMENT_PLAN_1.md)", "",
              f"Replay: `{a.diag}` (diagnose_ball_misses.py, fused chain = BallFuser.update). Main output "
              "conf >= 0.15, c5c threshold 0.70, specialist up to 15 frames after the last sighting.", ""]
    data, all_misses = {"splits": {}}, []
    for split in a.splits:
        keys = sorted(k for k in frames if k[0] == split)
        agg = new_agg()
        agg["seqs"] = len(keys)
        misses_s, runs_s = [], []
        for k in keys:
            rd = SeqReader(by_name[k[1]])
            labels = parse_labels(rd.labels())
            rd.close()
            st, ms, rl = analyse_seq(frames[k], labels, a.max_miss)
            for key in COUNT_KEYS:
                agg[key] += st[key]
            for nm in ("main", "fused"):
                for t in TOLS:
                    agg[nm][t][0] += st[nm][t][0]
                    agg[nm][t][1] += st[nm][t][1]
            for b, v in st["size"].items():
                for i in range(3):
                    agg["size"][b][i] += v[i]
            for m in ms:
                m["split"] = split
            misses_s += ms
            runs_s += rl
        L, out = split_block(split, agg, misses_s, runs_s)
        d_rec = diag_meta.get("splits", {}).get(split, {}).get("recall")
        if d_rec is not None and abs(d_rec - out["fused"]["recall"]["15"]) > 1e-6:
            L.append(f"**WARNING:** fused recall differs from diagnosis.json ({d_rec:.4f}).")
        report += L
        data["splits"][split] = out
        all_misses += misses_s
        print(f"[phase0] {split}: fused recall {out['fused']['recall']['15']:.2%}, {len(misses_s)} misses", flush=True)

    rows, findings, dup, names, clash = split_integrity(refs, a.frames_per_hash)
    pairs = sorted({(d["seq"], d["match_seq"]) for d in dup})
    data["split_integrity"] = {"counts": {s: len(v) for s, v in names.items()}, "test_name_clash": clash,
                               "clip_overlaps": findings, "near_duplicate_frames": len(dup),
                               "near_duplicate_pairs": pairs}
    report += ["## Split integrity", "",
               "Sequences per split: " + ", ".join(f"{s} {len(v)}" for s, v in names.items()) + ".", "",
               f"Test sequence names also in another split: {clash or 'none'}.", "",
               "Same game and overlapping clip time across splits (compared within one GSR source split, since "
               "`game_id` is not comparable across GSR train/valid): " + ("; ".join(findings) or "none") + ".", "",
               f"Near-duplicate frames (dHash Hamming <= 4, every {a.frames_per_hash}th frame) between select/test "
               f"and train/pool: {len(dup)}" + (" -- pairs: " + ", ".join(f"{s}~{m}" for s, m in pairs[:20]) if dup else "")
               + ".", "",
               "Static frames (black, scoreboard-only) can match by hash without any leak; inspect a pair before "
               "calling it one.", ""]

    (a.out / "report_phase0.md").write_text("\n".join(report) + "\n", encoding="utf-8")
    (a.out / "phase0.json").write_text(json.dumps(data, indent=1, default=float))
    if all_misses:
        with open(a.out / "buckets.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(all_misses[0]))
            w.writeheader()
            w.writerows(all_misses)
    with open(a.out / "split_integrity.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]) if rows else ["seq"])
        w.writeheader()
        w.writerows(rows)
    if dup:
        with open(a.out / "near_duplicates.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(dup[0]))
            w.writeheader()
            w.writerows(dup)
    print("\n".join(report), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/pipeline_improvement_1/phase3/convert_sntracking.py
"""
convert_sntracking.py -- SoccerNet-Tracking clips (MOT format: gameinfo.ini, gt/gt.txt, img1/) to the
GSR layout gsr_io.discover() reads (Labels-GameState.json + img1/), for Phase 3 of
docs/plans/PIPELINE_IMPROVEMENT_PLAN_1.md.

Why: tracking `test` (49 clips, games 7/8/11) is the GSR `test` split that c5c's `pool` was trained
with (82 clips); the Kaggle GSR dataset has only train/valid, so `pool` there is 33 clips. Writing
these under <out>/test/ makes discover() put them in `pool` (src split gsrtest).

Disk: labels only (~1 MB per clip); img1/ is a symlink to the input (no frames copied), so the
20 GB Kaggle working limit is not touched. Input may be extracted folders or the split zip; a zip
is extracted to --extract_dir (outside /kaggle/working).

Labels kept: ball (category 4) and persons (category 1) as bbox_image centre/size. No pitch lines.

    python scripts/pipeline_improvement_1/phase3/convert_sntracking.py --src <tracking root> --out <dir>
"""

from __future__ import annotations

import argparse
import configparser
import json
import os
import shutil
import zipfile
from pathlib import Path

BALL_CAT, PERSON_CAT = 4, 1


def _clips(src: Path, extract_dir: Path) -> list[Path]:
    clips = sorted({p.parent for p in src.rglob("gameinfo.ini") if (p.parent / "img1").is_dir()})
    if clips:
        return clips
    for zp in sorted(src.rglob("*.zip")):
        dest = extract_dir / zp.stem
        if not dest.exists():
            print(f"extracting {zp} -> {dest}", flush=True)
            with zipfile.ZipFile(zp) as z:
                z.extractall(dest)
        clips += sorted({p.parent for p in dest.rglob("gameinfo.ini") if (p.parent / "img1").is_dir()})
    return clips


def convert(clip: Path, out_root: Path) -> tuple[str, int]:
    cfg = configparser.ConfigParser()
    cfg.read_string(clip.joinpath("gameinfo.ini").read_text())
    s = cfg[cfg.sections()[0]]
    name = s["name"].strip()
    kind = {k.split("_")[1]: v.split(";")[0].strip() for k, v in s.items() if k.startswith("trackletid")}
    seq = configparser.ConfigParser()
    seq.read_string(clip.joinpath("seqinfo.ini").read_text())
    si = seq[seq.sections()[0]]
    W, H = int(si.get("imWidth", 1920)), int(si.get("imHeight", 1080))
    ext = si.get("imExt", ".jpg")
    files = sorted(p.name for p in (clip / "img1").iterdir() if p.suffix == ext)
    images = [{"image_id": int(Path(f).stem), "file_name": f, "width": W, "height": H} for f in files]
    anns, n_ball = [], 0
    for line in clip.joinpath("gt", "gt.txt").read_text().split():
        f, tid, x, y, w, h = line.split(",")[:6]
        k = kind.get(tid, "")
        if k == "ball":
            cat = BALL_CAT
            n_ball += 1
        elif k:
            cat = PERSON_CAT
        else:
            continue
        x, y, w, h = map(float, (x, y, w, h))
        anns.append({"image_id": int(f), "category_id": cat, "track_id": int(tid),
                     "bbox_image": {"x_center": x + w / 2, "y_center": y + h / 2, "w": w, "h": h}})
    d = out_root / name
    d.mkdir(parents=True, exist_ok=True)
    info = {"name": name, "im_dir": "img1", "game_id": s.get("gameID", "").strip(),
            "clip_start": s.get("clipStart", "").strip(), "clip_stop": s.get("clipStop", "").strip(),
            "game_time_start": s.get("gameTimeStart", "").strip(), "source": "SoccerNet-Tracking"}
    (d / "Labels-GameState.json").write_text(json.dumps({"info": info, "images": images, "annotations": anns}))
    link = d / "img1"
    if not link.exists():
        try:
            os.symlink((clip / "img1").resolve(), link, target_is_directory=True)
        except OSError:                     # Windows without symlink rights (local smoke tests): copy
            shutil.copytree(clip / "img1", link)
    return name, n_ball


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--src", type=Path, required=True, help="tracking dataset root (folders or zips)")
    ap.add_argument("--out", type=Path, required=True, help="GSR-layout root; clips go to <out>/test/")
    ap.add_argument("--extract_dir", type=Path, default=Path("/tmp/sntracking"))
    ap.add_argument("--split_dir", default="test", help="only clips under a folder of this name")
    a = ap.parse_args()
    clips = [c for c in _clips(a.src, a.extract_dir) if a.split_dir in c.parts]
    tot = 0
    for c in clips:
        _, n = convert(c, a.out / "test")
        tot += n
    print(f"converted {len(clips)} clips, {tot} ball boxes -> {a.out / 'test'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kit/cv_service/scripts/pipeline_improvement_1/phase3/phase3a_mine.py
"""
phase3a_mine.py -- Phase 3a of docs/plans/PIPELINE_IMPROVEMENT_PLAN_1.md: replay the live system
(Phase 1: plausibility gate + c5c threshold 0.45) over every split, keep the specialist's top heatmap
peaks per call, mine the specialist's own failures for training, and test multi-peak selection.

Stages:
  replay     (GPU, shardable) per sequence: the Phase 1 replay, identical to phase1_sweep.simulate,
             plus per specialist call the top-K local peaks (max-pool NMS) and per frame the main
             detector's candidates. -> <out>/frames/<split>/<seq>.jsonl
  multipeak  (GPU) on select, then frozen on test: when c5c runs, accept the highest peak
             >= thr_lo inside the Phase 1 gate radius (r0 + speed * gap of the last sighting), else
             the top peak if >= hi. Grid over (thr_lo, hi) on select, objective = recall with
             precision >= Phase 1 - 1 pp; then test against Phase 1 (kill criterion +0.5 pp / -1 pp).
  report     (CPU) per-split Phase 1 numbers and miss buckets (phase0_report.analyse_seq), the mined
             set from train + pool, and the multi-peak result.
             -> report_phase3a.md, phase3a.json, mined.csv, frames.jsonl (select + test, diagnose
             format for phase0_report.py's split-integrity check)

Mined rows (train + pool only; select/test are never mined), one per example:
  hard_pos   ball inside the runtime window but c5c missed it: bucket B5 (peak < thr) or B6 (output
             > 15 px off), and B3 frames whose window (placed by a wrong sighting) still held the ball
  neg_spec   a c5c peak >= --neg_min more than --neg_dist px from the labelled ball (frames with a
             label only: there the ball is known to be elsewhere)
  neg_main   a main-detector ball >= 0.15 more than --neg_dist px from the labelled ball
Fields: split, seq, src (gsr | sntracking), t, kind, bucket, x, y (example position, full frame),
x0, y0 (runtime window, or -1), conf, gt_x, gt_y, gt_size, gap. Capped per sequence and kind
(--cap, strongest first) so one bad clip cannot dominate.

    python scripts/pipeline_improvement_1/phase3/phase3a_mine.py replay --src <roots> --cache <main cache> \
        --ckpt c5c_best.pt --out <dir> [--shard i --nshards n --device cuda:i]
    python scripts/pipeline_improvement_1/phase3/phase3a_mine.py multipeak --src ... --cache ... --ckpt ... --out <dir>
    python scripts/pipeline_improvement_1/phase3/phase3a_mine.py report --src ... --out <dir>
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve()
ROOT = HERE.parents[3]
for p in (ROOT / "gsr_pipeline", ROOT / "scripts" / "ball_temporal", ROOT / "scripts" / "ball_eval",
          ROOT / "scripts" / "pipeline_improvement_1" / "phase1", ROOT / "scripts" / "pipeline_improvement_1" / "phase0"):
    sys.path.insert(0, str(p))
from ball_gate import BallGate, GateParams  # noqa: E402
from gsr_io import SeqReader, discover, parse_labels  # noqa: E402

GATE = GateParams(enabled=True, r0=30.0, speed=50.0, conf_hi=0.6, max_rejects=15)    # Phase 1, live
THR = 0.45
MATCH = 15.0
CROP = 640
SPLITS = ("train", "pool", "select", "test")
MP_THR_LO = (0.20, 0.25, 0.30, 0.35, 0.45)
MP_HI = (0.45, 0.55, 0.70, 1.01)


def _src_of(ref) -> str:
    return "sntracking" if ref.name.startswith("SNMOT") else "gsr"


def _d(p, gts):
    return min((float(np.hypot(p[0] - g[0], p[1] - g[1])) for g in gts), default=np.inf) if p else np.inf


# ---------------------------------------------------------------- GPU side

def make_peak_specialist(ckpt, dev, topk, peak_min, nms):
    import torch
    import torch.nn.functional as F
    from eval_ball_temporal import Specialist, build_input, decode_hm, load_model
    from heatmap_model import STRIDE

    class PeakSpecialist(Specialist):
        """Specialist whose memo also keeps the top-K local heatmap peaks of each window."""

        def __init__(self, model, channels, dev):
            super().__init__(model, channels, dev)
            self.peaks: dict = {}

        @torch.no_grad()
        def __call__(self, key, frames, x0, y0):
            if key in self.memo:
                return self.memo[key]
            x = np.concatenate([f[y0:y0 + CROP, x0:x0 + CROP, ::-1] for f in frames], axis=2)
            x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))[None].to(self.dev)
            t0 = time.perf_counter()
            with torch.autocast("cuda", enabled=self.dev.type == "cuda"):
                logits, off = self.model(build_input(x, self.channels))
            p = decode_hm(logits, off)[0].tolist()
            hm = logits.float().sigmoid()[0, 0]
            S = hm.shape[-1]
            keep = (hm == F.max_pool2d(hm[None, None], nms, 1, nms // 2)[0, 0]) & (hm >= peak_min)
            conf, idx = hm[keep].flatten(), keep.flatten().nonzero()[:, 0]
            order = conf.argsort(descending=True)[:topk]
            o = off.float()[0].view(2, -1)
            pk = []
            for i in idx[order].tolist():
                iy, ix = divmod(i, S)
                pk.append((round((ix + float(o[0, i])) * STRIDE + x0, 1), round((iy + float(o[1, i])) * STRIDE + y0, 1),
                           round(float(hm.flatten()[i]), 4)))
            if self.dev.type == "cuda":
                torch.cuda.synchronize()
            self.ms += (time.perf_counter() - t0) * 1000
            self.calls += 1
            self.memo[key] = (p[0] + x0, p[1] + y0, p[2])
            self.peaks[key] = pk
            return self.memo[key]

    model, channels, epoch = load_model(ckpt, dev)
    return PeakSpecialist(model, channels, dev), epoch


def replay(sd, spec, a, mp=None) -> list[dict]:
    """phase1_sweep.simulate (Phase 1 as wired) + peaks and main candidates per frame.
    mp = (thr_lo, hi): multi-peak selection instead of the top peak."""
    from eval_ball_temporal import clamp_window
    gate = BallGate(GATE, a.max_miss)
    out, last = [], None
    for t in sorted(sd.files)[: a.max_frames or None]:
        cands = sorted((b for b in sd.dets.get(t, []) if b[4] >= a.main_conf), key=lambda b: -b[4])
        rec = dict(t=t, out=None, src="", win=None, sconf=None, gap=None, rej=None, peaks=None,
                   main=[[round(b[0], 1), round(b[1], 1), round(b[4], 3)] for b in cands])
        i = gate.pick([(b[0], b[1], b[4]) for b in cands], t, last)
        if i is not None:
            last = (cands[i][0], cands[i][1], t)
            rec.update(out=(cands[i][0], cands[i][1]), src="main")
            out.append(rec)
            continue
        if cands:
            rec["rej"] = (cands[0][0], cands[0][1])
        if last is not None and t - last[2] <= a.max_miss:
            x0, y0, _ = clamp_window(last[0], last[1], sd.W, sd.H)
            gap = t - last[2]
            rec.update(gap=gap, win=(int(x0), int(y0)))
            key = (sd.ref.name, t, x0, y0)
            px, py, conf = spec(key, None if key in spec.memo else sd.window_frames(t), x0, y0)
            rec["sconf"] = float(conf)
            rec["peaks"] = spec.peaks.get(key, [])
            pick = None
            if mp is None:
                if conf >= THR:
                    pick = (px, py)
            else:
                lo, hi = mp
                allowed = GATE.r0 + GATE.speed * max(gap, 1)
                pick = next(((x, y) for x, y, c in rec["peaks"]
                             if c >= lo and np.hypot(x - last[0], y - last[1]) <= allowed), None)
                if pick is None and conf >= hi:
                    pick = (px, py)
            if pick is not None:
                rec.update(out=pick, src="spec")
                last = (pick[0], pick[1], t)
        out.append(rec)
    return out


def _load(a, splits):
    from eval_ball_temporal import SeqData
    refs = [r for r in discover(a.src) if r.split in splits]
    if a.max_seqs:
        refs = [r for s in splits for r in [q for q in refs if q.split == s][: a.max_seqs]]
    return refs, SeqData


def stage_replay(a):
    import torch
    dev = torch.device(a.device if torch.cuda.is_available() else "cpu")
    spec, epoch = make_peak_specialist(a.ckpt, dev, a.topk, a.peak_min, a.nms)
    refs, SeqData = _load(a, a.splits)
    refs = [r for i, r in enumerate(refs) if i % a.nshards == a.shard]
    print(f"[shard {a.shard}] c5c epoch {epoch} on {dev}; {len(refs)} sequences", flush=True)
    t_all = time.time()
    for k, ref in enumerate(refs, 1):
        dst = a.out / "frames" / ref.split / f"{ref.name}.jsonl"
        if dst.exists():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        t0, c0 = time.time(), spec.calls
        sd = SeqData(ref, a.cache)
        recs = replay(sd, spec, a)
        sd.reader.close()
        spec.memo.clear()
        spec.peaks.clear()
        tmp = dst.with_suffix(".tmp")
        with open(tmp, "w") as f:
            for r in recs:
                r.update(split=ref.split, seq=ref.name, gt=[list(g) for g in sd.gt.get(r["t"], [])])
                f.write(json.dumps(r) + "\n")
        tmp.replace(dst)
        print(f"[shard {a.shard}] {k}/{len(refs)} {ref.split}/{ref.name}: {len(recs)} frames, "
              f"{spec.calls - c0} c5c calls, {time.time() - t0:.0f}s, elapsed {(time.time() - t_all) / 60:.1f} min",
              flush=True)


def stage_multipeak(a):
    import torch
    import phase1_sweep
    dev = torch.device(a.device if torch.cuda.is_available() else "cpu")
    spec, _ = make_peak_specialist(a.ckpt, dev, a.topk, a.peak_min, a.nms)
    res = {}
    for role, split in (("select", a.search_split), ("test", a.eval_split)):
        refs, SeqData = _load(a, (split,))
        sds = [SeqData(r, a.cache) for r in refs]
        base = phase1_sweep.score(sds, [replay(sd, spec, a) for sd in sds])
        floor = base["precision"] - a.prec_drop
        rows = [dict(thr_lo=None, hi=None, **base)]
        print(f"[{role}:{split}] phase1: recall {base['recall']:.2%} precision {base['precision']:.2%}", flush=True)
        grid = [(lo, hi) for lo in MP_THR_LO for hi in MP_HI if hi >= lo] if role == "select" else [res["chosen"]]
        for lo, hi in grid:
            s = phase1_sweep.score(sds, [replay(sd, spec, a, (lo, hi)) for sd in sds])
            rows.append(dict(thr_lo=lo, hi=hi, **s))
            print(f"[{role}:{split}] multipeak lo {lo:.2f} hi {hi:.2f}: recall {s['recall']:.2%} precision {s['precision']:.2%}",
                  flush=True)
        if role == "select":
            ok = [r for r in rows[1:] if r["precision"] >= floor]
            best = max(ok, key=lambda r: (r["recall"], r["precision"])) if ok else None
            res["chosen"] = (best["thr_lo"], best["hi"]) if best else (THR, THR)
        res[role] = rows
        for sd in sds:
            sd.reader.close()
    b, t = res["test"][0], res["test"][1]
    res["keep"] = t["recall"] - b["recall"] >= a.min_gain and b["precision"] - t["precision"] <= a.prec_drop
    (a.out / "multipeak.json").write_text(json.dumps(res, indent=1, default=float))
    print(f"chosen on select {res['chosen']}; test {t['recall']:.2%} / {t['precision']:.2%} vs phase1 "
          f"{b['recall']:.2%} / {b['precision']:.2%}: {'KEEP' if res['keep'] else 'REVERT'}", flush=True)


# ---------------------------------------------------------------- CPU side

def mine_seq(rows, labels, ref, a):
    import phase0_report
    _im, _f, balls, *_ = labels
    _st, misses, _runs = phase0_report.analyse_seq(rows, labels, a.max_miss)
    bucket = {m["t"]: m["bucket"] for m in misses}
    by_t = {r["t"]: r for r in rows}
    out = defaultdict(list)
    for t, b in bucket.items():
        r = by_t[t]
        if b not in ("B3", "B5", "B6") or r["win"] is None:
            continue
        g = r["gt"][0]
        x0, y0 = r["win"]
        if not (x0 <= g[0] < x0 + CROP and y0 <= g[1] < y0 + CROP):
            continue
        size = max(balls[t][0][2], balls[t][0][3]) if balls.get(t) else 0.0
        near = max((c for x, y, c in r["peaks"] or [] if np.hypot(x - g[0], y - g[1]) <= MATCH), default=0.0)
        out["hard_pos"].append(dict(kind="hard_pos", bucket=b, t=t, x=g[0], y=g[1], x0=x0, y0=y0, conf=near,
                                    gt_x=g[0], gt_y=g[1], gt_size=size, gap=r["gap"]))
    for r in rows:
        if not r["gt"]:
            continue
        g = r["gt"][0]
        size = max(balls[r["t"]][0][2], balls[r["t"]][0][3]) if balls.get(r["t"]) else 0.0
        for x, y, c in r["peaks"] or []:
            if c >= a.neg_min and np.hypot(x - g[0], y - g[1]) > a.neg_dist:
                out["neg_spec"].append(dict(kind="neg_spec", bucket="", t=r["t"], x=x, y=y, x0=r["win"][0], y0=r["win"][1],
                                            conf=c, gt_x=g[0], gt_y=g[1], gt_size=size, gap=r["gap"]))
        for x, y, c in r["main"]:
            if np.hypot(x - g[0], y - g[1]) > a.neg_dist:
                out["neg_main"].append(dict(kind="neg_main", bucket="", t=r["t"], x=x, y=y, x0=-1, y0=-1, conf=c,
                                            gt_x=g[0], gt_y=g[1], gt_size=size, gap=""))
    rows_out = []
    for k, v in out.items():
        v.sort(key=lambda e: -e["conf"] if k != "hard_pos" else e["conf"])    # hard_pos: the worst misses first
        rows_out += [dict(split=ref.split, seq=ref.name, src=_src_of(ref), **e) for e in v[: a.cap]]
    return rows_out, misses


def stage_report(a):
    import phase0_report
    import phase1_sweep
    refs = {r.name: r for r in discover(a.src)}
    per_split, buckets, mined = defaultdict(list), defaultdict(Counter), []
    nball = Counter()
    fj = open(a.out / "frames.jsonl", "w")
    for split in SPLITS:
        for p in sorted((a.out / "frames" / split).glob("*.jsonl")):
            ref = refs[p.stem]
            rows = [json.loads(line) for line in p.read_text().splitlines()]
            rd = SeqReader(ref)
            labels = parse_labels(rd.labels())
            rd.close()
            for r in rows:
                r["out"] = tuple(r["out"]) if r["out"] else None
            grp = f"{split}/{_src_of(ref)}" if split == "pool" else split
            per_split[grp].append((ref, rows))
            nball[grp] += sum(bool(r["gt"]) for r in rows)
            if split in a.mine_splits:
                m, misses = mine_seq(rows, labels, ref, a)
                mined += m
            else:
                _st, misses, _ = phase0_report.analyse_seq(rows, labels, a.max_miss)
                for r in rows:
                    fj.write(json.dumps(dict(split=split, seq=ref.name, t=r["t"], gt=r["gt"], out=r["out"], src=r["src"],
                                             win=r["win"], sconf=r["sconf"], gap=r["gap"],
                                             hit=float(bool(r["gt"]) and _d(r["out"], r["gt"]) <= MATCH))) + "\n")
            for m in misses:
                buckets[grp][m["bucket"]] += 1
    fj.close()

    class _SD:                       # phase1_sweep.score needs .ref.name and .gt only
        def __init__(self, ref, rows):
            self.ref, self.gt = ref, {r["t"]: [tuple(g) for g in r["gt"]] for r in rows if r["gt"]}

    summ = {}
    for grp, items in per_split.items():
        s = phase1_sweep.score([_SD(ref, rows) for ref, rows in items], [rows for _, rows in items])
        summ[grp] = dict(seqs=len(items), **s, buckets=dict(buckets[grp]))
    cols = ["split", "seq", "src", "t", "kind", "bucket", "x", "y", "x0", "y0", "conf", "gt_x", "gt_y", "gt_size", "gap"]
    with open(a.out / "mined.csv", "w", newline="") as f:
        w = csv.DictWriter(f, cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(mined)
    mp = json.loads((a.out / "multipeak.json").read_text()) if (a.out / "multipeak.json").exists() else None
    mc = Counter((m["split"] + "/" + m["src"], m["kind"]) for m in mined)
    hb = Counter((m["split"] + "/" + m["src"], m["bucket"]) for m in mined if m["kind"] == "hard_pos")
    json.dump({"phase1": summ, "mined": {f"{k[0]}|{k[1]}": v for k, v in mc.items()}, "multipeak": mp},
              open(a.out / "phase3a.json", "w"), indent=1, default=float)

    pc = lambda x: f"{x:.2%}"  # noqa: E731
    L = ["# Phase 3a: Phase 1 replay over every split, mined failures, multi-peak selection", "",
         "## Phase 1 (live system) per split", "",
         "train and pool are c5c's training data, so their numbers are optimistic; select and test are held out.", "",
         "| split | seqs | ball frames | recall @15 | precision | B3 | B5 | B6 | B7 | B1 |", "|---|---|---|---|---|---|---|---|---|---|"]
    for grp in sorted(summ):
        s, bk = summ[grp], summ[grp]["buckets"]
        n = max(nball[grp], 1)
        L.append(f"| {grp} | {s['seqs']} | {nball[grp]} | {pc(s['recall'])} | {pc(s['precision'])} | "
                 + " | ".join(pc(bk.get(b, 0) / n) for b in ("B3", "B5", "B6", "B7", "B1")) + " |")
    L += ["", "Buckets as share of ball frames (phase0_report definitions).", "",
          f"## Mined set (`mined.csv`, cap {a.cap} per sequence and kind)", "",
          "| source | hard_pos | neg_spec | neg_main |", "|---|---|---|---|"]
    for grp in sorted({k[0] for k in mc}):
        L.append(f"| {grp} | " + " | ".join(str(mc.get((grp, k), 0)) for k in ("hard_pos", "neg_spec", "neg_main")) + " |")
    L += ["", "hard_pos by bucket: " + ", ".join(f"{g} {b}: {v}" for (g, b), v in sorted(hb.items())), ""]
    if mp:
        b, t = mp["test"][0], mp["test"][1]
        L += ["## Multi-peak selection", "",
              f"Chosen on select: thr_lo {mp['chosen'][0]}, hi {mp['chosen'][1]} (grid in `multipeak.json`).", "",
              "| test | recall @15 | precision | longest miss |", "|---|---|---|---|",
              f"| phase1 | {pc(b['recall'])} | {pc(b['precision'])} | {b['longest_miss_frames']} |",
              f"| multi-peak | {pc(t['recall'])} | {pc(t['precision'])} | {t['longest_miss_frames']} |", "",
              f"**Verdict: {'KEEP' if mp['keep'] else 'REVERT'}** (kill criterion +{a.min_gain:.1%} recall, "
              f"<= {a.prec_drop:.0%} precision loss)."]
    (a.out / "report_phase3a.md").write_text("\n".join(L) + "\n", encoding="utf-8")
    print("\n".join(L))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", choices=["replay", "multipeak", "report"])
    ap.add_argument("--src", required=True, help="GSR root(s), comma-separated (add the converted tracking clips)")
    ap.add_argument("--cache", type=Path, help="main-detector cache (replay, multipeak)")
    ap.add_argument("--ckpt", type=Path, help="c5c checkpoint (replay, multipeak)")
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--splits", nargs="+", default=list(SPLITS), choices=SPLITS)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--nshards", type=int, default=1)
    ap.add_argument("--max_seqs", type=int, default=0, help="per split (smoke tests)")
    ap.add_argument("--max_frames", type=int, default=0)
    ap.add_argument("--max_miss", type=int, default=15)
    ap.add_argument("--main_conf", type=float, default=0.15)
    ap.add_argument("--topk", type=int, default=5)
    ap.add_argument("--peak_min", type=float, default=0.05)
    ap.add_argument("--nms", type=int, default=5, help="max-pool size for local peaks (heatmap cells)")
    ap.add_argument("--neg_min", type=float, default=0.2)
    ap.add_argument("--neg_dist", type=float, default=40.0)
    ap.add_argument("--mine_splits", nargs="+", default=["train", "pool"], help="never select/test except smoke tests")
    ap.add_argument("--cap", type=int, default=150)
    ap.add_argument("--search_split", default="select", help="multipeak: where (thr_lo, hi) is chosen")
    ap.add_argument("--eval_split", default="test", help="multipeak: frozen evaluation")
    ap.add_argument("--prec_drop", type=float, default=0.01)
    ap.add_argument("--min_gain", type=float, default=0.005)
    a = ap.parse_args()
    a.out.mkdir(parents=True, exist_ok=True)
    {"replay": stage_replay, "multipeak": stage_multipeak, "report": stage_report}[a.stage](a)


if __name__ == "__main__":
    main()

## 1. Environment

Installs `ultralytics>=8.4` (YOLO26) and lists the GPUs.

In [ ]:
STREAM([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4"])
import torch
NGPU = max(torch.cuda.device_count(), 1)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. SoccerNet-Tracking test -> GSR layout (pool 33 -> 82)

Labels only; each clip's `img1/` is a symlink into the input. Expect train 57 / pool 82 / select 10 / test 15.

In [ ]:
STREAM([sys.executable, "-u", f"{KIT}/scripts/pipeline_improvement_1/phase3/convert_sntracking.py",
        "--src", TRACK_INPUT, "--out", SNT, "--extract_dir", "/tmp/sntracking"])
sys.path.insert(0, f"{KIT}/scripts/ball_temporal")
from gsr_io import discover
REFS = discover(SRC)
print({s: sum(r.split == s for r in REFS) for s in ("train", "pool", "select", "test")},
      "| pool from tracking:", sum(r.name.startswith("SNMOT") for r in REFS))
SMOKE_SEQS = [next(r.name for r in REFS if r.split == s) for s in ("train", "pool", "select", "test")] + \
             [next(r.name for r in REFS if r.name.startswith("SNMOT"))]
DISK()

## 3. Main-detector cache (all splits)

Ball candidates (top 5, conf >= 0.05) per frame, YOLO26s at imgsz 1280, FP16, one process per GPU. Resumable.

In [ ]:
base = [sys.executable, "-u", f"{KIT}/scripts/ball_temporal/cache_main_detections.py", "run", "--src", SRC,
        "--out", CACHE, "--weights", MAIN, "--splits", "train", "pool", "select", "test", "--half"]
if SMOKE:
    base += ["--seqs", *SMOKE_SEQS, "--max_frames", "150"]
PARALLEL([base + ["--device", str(i), "--shard", str(i), "--nshards", str(NGPU)] for i in range(NGPU)],
         [WORK / f"cache_gpu{i}.log" for i in range(NGPU)])
print({s: len(list((CACHE / s).glob("*.json"))) for s in ("train", "pool", "select", "test")})
DISK()

## 4. Phase 1 replay with heatmap peaks (all splits)

`phase3a_mine.py replay`: per sequence, identical to the live system, plus c5c's top-5 peaks and the main candidates.

In [ ]:
base = [sys.executable, "-u", f"{KIT}/scripts/pipeline_improvement_1/phase3/phase3a_mine.py", "replay",
        "--src", SRC, "--cache", CACHE, "--ckpt", C5C, "--out", WORK]
if SMOKE:
    base += ["--max_seqs", "1", "--max_frames", "150"]
PARALLEL([base + ["--device", f"cuda:{i}", "--shard", str(i), "--nshards", str(NGPU)] for i in range(NGPU)],
         [WORK / f"replay_gpu{i}.log" for i in range(NGPU)])

## 5. Multi-peak selection: chosen on select, frozen on test

Grid over (thr_lo, hi); kill criterion as Phase 1 (+0.5 pp recall, <= 1 pp precision loss).

In [ ]:
STREAM([sys.executable, "-u", f"{KIT}/scripts/pipeline_improvement_1/phase3/phase3a_mine.py", "multipeak",
        "--src", SRC, "--cache", CACHE, "--ckpt", C5C, "--out", WORK]
       + (["--max_seqs", "1", "--max_frames", "150"] if SMOKE else []))

## 6. Report and mined set

Per-split Phase 1 numbers and buckets (pool split into GSR and tracking clips), `mined.csv`, multi-peak verdict.

In [ ]:
STREAM([sys.executable, "-u", f"{KIT}/scripts/pipeline_improvement_1/phase3/phase3a_mine.py", "report",
        "--src", SRC, "--out", WORK])
SHOW(WORK / "report_phase3a.md")

## 7. Split integrity with the new clips

`phase0_report.py` on the select/test replay: same game + overlapping clip time across splits, and near-duplicate frames (dHash, Hamming <= 4, every 25th frame) between {select, test} and {train, pool}.

In [ ]:
STREAM([sys.executable, "-u", f"{KIT}/scripts/pipeline_improvement_1/phase0/phase0_report.py",
        "--src", SRC, "--diag", WORK, "--out", WORK / "integrity", "--splits", "select", "test",
        "--frames_per_hash", "100" if SMOKE else "25"])
txt = (WORK / "integrity" / "report_phase0.md").read_text()
print(txt[txt.find("## Split integrity"):])

## 8. Deliverables

Packs the work folder (reports, mined set, replay frames, caches) into `/kaggle/working/phase3a_deliverables.zip`.

In [ ]:
import hashlib, json, zipfile
out = Path("/kaggle/working/phase3a_deliverables.zip")
manifest = {}
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(p for p in WORK.rglob("*") if p.is_file()):
        rel = f.relative_to(WORK).as_posix()
        zf.write(f, rel)
        manifest[rel] = hashlib.sha256(f.read_bytes()).hexdigest()
    zf.writestr("MANIFEST.json", json.dumps(manifest, indent=1))
print(f"{out}: {len(manifest)} files, {out.stat().st_size / 1e6:.1f} MB")
DISK()